# 07 · Species impacts — GIDIAS records, LLM gap-filling and the master file

**Data stream:** documented impact of each species where it is already
alien (EICAT, SEICAT and context).
**Output feeds:** the viewer only. `species_impacts_master.csv` is
imported next to the lists and joined on `gbif_id` in the browser.

> **Impact never enters the risk model.** Nothing written here is read by
> notebooks 01–06. The risk score says how likely a species is to
> establish; this notebook adds what is known about the harm it has done
> elsewhere, as information for the experts who assess the lists.

---

## What this notebook does

1. **GIDIAS (primary source).** Reads the *Global Impacts Dataset of
   Invasive Alien Species* (about 22 900 impact records from the
   literature), gives every species the GBIF Backbone species key of the
   pipeline, and codes every record as an **EICAT** category (harm to
   native species) and a **SEICAT** category (harm to human well-being).
2. **LLM (gap filler).** For the top `TOP_X` species of the exported
   lists that GIDIAS does not cover, a language model is asked for the
   same fields, in the same vocabulary, so both sources fit one table.
3. **Benchmark.** The same model scores species that GIDIAS *does*
   cover; the agreement tells you how far to trust its answers.
4. **Master file.** One row per species with the same columns whatever
   the source, and the source of the EICAT block, the SEICAT block and
   the context written next to them.

> **Neither source is an IUCN assessment.** A GIDIAS category is the
> highest magnitude that any single publication reports. An LLM category
> is what a model recalls or reads from the evidence it is given; it can
> be confidently wrong and can cite sources that do not exist. Both are
> there to decide which species to look at first.

## The two classifications in brief

| Code | EICAT: impact on native species | SEICAT: impact on people's activities |
|---|---|---|
| MC | no reduction in the performance of native individuals | no change in activities |
| MN | performance of native individuals reduced | activities more difficult, nobody stops |
| MO | native population decline | fewer people take part in an activity |
| MR | local extinction, reversible | activity disappears locally, reversible |
| MV | local extinction, irreversible | activity disappears locally, irreversible |
| DD | impact data insufficient | impact data insufficient |

Both take the **highest** impact documented anywhere in the alien range.

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Sources, configuration, plug in a model, helpers | — |
| 1 | Read GIDIAS, tidy the fields | `g` |
| 2 | Match names to GBIF Backbone species keys | `names` |
| 3 | Code every record: EICAT, SEICAT, positive impacts | `records` |
| 4 | Summarise GIDIAS per species | `gidias` |
| 5 | Read the lists, select the species GIDIAS does not cover | `todo` |
| 6 | Prompt and answer format (GIDIAS structure) | functions |
| 7 | Test the model on one species | one answer |
| 8 | Score the species (cached, resumable) | `llm` |
| 9 | Benchmark the model against GIDIAS | `benchmark` |
| 10 | Merge both sources into the master file | `master` |
| 11 | Quality checks | `summary` |
| 12 | Write outputs | files |

## Outputs

| File | Content |
|---|---|
| `data/species_impacts_master.csv` | **the master file**: one row per species, both sources, same columns. A copy goes to `model_outputs/exports/`, so importing that folder in the viewer picks it up |
| `data/gidias_impact_records.csv` | one row per GIDIAS record, tidied, with `gbif_id` and its categories |
| `data/gidias_name_matching_audit.csv` | every GIDIAS name with its match result |
| `data/impact_llm_scores.csv` | every LLM-scored species with the category of each single answer |
| `data/model_outputs/impact_llm_benchmark.csv` | benchmark species: GIDIAS next to the LLM |
| `data/model_outputs/impacts_summary.json` | settings, counts and agreement; also copied into `model_card.json` |
| `data/cache/impact_llm_responses.jsonl` | raw model answers (re-runs cost nothing) |

Column definitions: `species_impacts_metadata.md`.

## Design rules (read before editing)

1. **GIDIAS first, field by field.** The EICAT block comes from GIDIAS
   when GIDIAS has a category MC–MV, otherwise from the LLM if it gives
   one, otherwise it is DD. The SEICAT block is decided the same way,
   independently. `eicat_source` and `seicat_source` say which.
2. **One vocabulary.** The LLM must answer with the GIDIAS terms
   (mechanisms, affected taxa, well-being constituents, NCP categories,
   realms), so the two sources can be filtered together.
3. **EICAT counts harm to native species only.** Positive impacts,
   changes of ecosystem properties without an affected native taxon
   (`NATIVE_TAXA_ONLY`) and impacts on people are kept in their own
   columns.
4. **GIDIAS does not code reversibility.** A local extinction is stored
   as MR (`LEVEL3_CATEGORY`); only a recorded global extinction is MV.
5. **Same species key as everywhere else:** the GBIF Backbone species
   key, with subspecies and synonyms rolled up as in notebook 03.
6. **No leak in the benchmark.** GIDIAS text is given to the model as
   evidence only for species GIDIAS could not categorise, never for
   benchmark species.

## How to run

1. Install (once): `pip install pandas numpy requests tqdm`
   (`country_converter` is optional and adds ISO3 codes).
2. Put the GIDIAS file in `../data/` (Step 0). Run notebook 03 first so
   its species keys are reused; export lists with notebook 05 or 06 if
   you want the LLM part.
3. **First run with `PROVIDER = "dry_run"`** (the default). The GIDIAS
   part is real and the master file is written from GIDIAS alone; the
   LLM steps run on made-up answers that are **not** merged, so you can
   see them work.
4. Plug in a model (Step 0), set `PROVIDER` and `MODEL`, test it in
   Step 7, then *Run All*. Step 5 prints how many model calls will be
   made.
5. In the viewer: import `model_outputs/exports/` (or drop
   `species_impacts_master.csv` next to the lists).

## Step 0 · Data sources and configuration

| Input | Source | How to obtain |
|---|---|---|
| `GIDIAS_v1.1_20260723_machine_read.csv` | GIDIAS v1.1 (file dated 2026-07-23) — Bacher S, Ryan-Colton E, Coiro M, et al. (2025) Global Impacts Dataset of Invasive Alien Species (GIDIAS). *Scientific Data* 12: 832, [doi:10.1038/s41597-025-05184-5](https://doi.org/10.1038/s41597-025-05184-5). Licence CC BY 4.0 | **Manual download** from the repository named under "Data Records" in the paper; use the *machine-read* csv. Note the repository DOI and download date in `species_impacts_metadata.md`. |
| GBIF Backbone Taxonomy | GBIF Secretariat, [doi:10.15468/39omei](https://doi.org/10.15468/39omei) | **Automatic**, `https://api.gbif.org/v1/species/match`, only for names notebook 03 has not matched |
| `model_outputs/exports/scan_<location>.csv` *(for the LLM part)* | notebook 05 or 06 | run one of them |
| `eicat_evidence/<Genus_species>.txt` *(optional)* | your own notes or abstracts per species | **Manual**; given to the model as evidence |
| `eicat_lookup.csv` *(optional)* | published IUCN EICAT assessments, <https://www.iucngisd.org/gisd/> | **Manual entry** (`gbif_id`, `eicat`); shown as a reference column `eicat_expert`, never used to change a category |
| a language model *(optional)* | any | see "Plug in a model" |

GIDIAS was compiled for chapter 4 of the IPBES assessment of invasive
alien species. Each row is one impact reported by one source.
SEICAT: Bacher S, Blackburn TM, Essl F, et al. (2018) Socio-economic
impact classification of alien taxa (SEICAT). *Methods in Ecology and
Evolution* 9: 159–168. EICAT: IUCN (2020) *IUCN EICAT Categories and
Criteria*, first edition.

In [3]:
import hashlib
import json
import os
import re
import shutil
import time
import urllib.error
import urllib.request
import warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

### Configuration

Paths and every choice that changes a category live here. For a first
run you only change `DATA` and `CONTACT_EMAIL`.

In [4]:
# ---------------------------------------------------------------- paths
# Set the environment variable IAS_DATA_DIR to use another data folder.

load_dotenv(Path.cwd().parent / ".env")

DATA = Path(os.environ.get("IAS_DATA_DIR", Path.cwd().parent / "data"))
GIDIAS_CSV = DATA / "GIDIAS_v1.1_20260723_machine_read.csv"
GIDIAS_VERSION = "GIDIAS v1.1"  # written into every row it produces
TAXONOMY_CSV = DATA / "taxonomy_backbone.csv"  # notebook 03
NAME_AUDIT_CSV = DATA / "name_matching_audit.csv"  # notebook 03
NAME_CACHE = DATA / "cache" / "gbif_name_matches.jsonl"  # shared with 03
EXPORT_DIR = DATA / "model_outputs" / "exports"  # lists of 05 / 06
EVIDENCE_DIR = DATA / "eicat_evidence"  # optional notes per species
EXPERT_CSV = DATA / "eicat_lookup.csv"  # optional IUCN assessments

# ---------------------------------------------------------------- outputs
MASTER_NAME = "species_impacts_master.csv"
MASTER_OUT = DATA / MASTER_NAME
RECORDS_OUT = DATA / "gidias_impact_records.csv"
MATCH_AUDIT_OUT = DATA / "gidias_name_matching_audit.csv"
LLM_SCORES = DATA / "impact_llm_scores.csv"
LLM_CACHE = DATA / "cache" / "impact_llm_responses.jsonl"
BENCHMARK_OUT = DATA / "model_outputs" / "impact_llm_benchmark.csv"
SUMMARY_OUT = DATA / "model_outputs" / "impacts_summary.json"
BACKUP_PREVIOUS = True  # keep overwritten files as *_previous.*
COPY_TO_EXPORTS = True  # copy the master file next to the lists

# ---------------------------------------------------------------- GBIF API
USE_GBIF_API = True  # False: only names known to notebook 03 are used
GBIF_MATCH_URL = "https://api.gbif.org/v1/species/match"
CONTACT_EMAIL = os.getenv("GBIF_EMAIL")   # <- set this
MAX_CONCURRENT = 8
GBIF_TIMEOUT = 30  # seconds
GBIF_RETRIES = 4
MIN_FUZZY_CONFIDENCE = 90  # as in notebook 03
SPECIES_RANKS = {"SPECIES", "SUBSPECIES", "VARIETY", "FORM",
                 "INFRASPECIFIC_NAME", "INFRASUBSPECIFIC_NAME",
                 "SUBVARIETY", "SUBFORM"}

# ---------------------------------------------------------------- coding
# EICAT category per GIDIAS magnitude of a negative impact on nature:
# 0 no reduction in performance, 1 performance reduced, 2 population
# decline, 3 local (or global) extinction.
LEVEL_TO_EICAT = {0: "MC", 1: "MN", 2: "MO"}
# GIDIAS does not say whether a local extinction is reversible (MR) or
# not (MV). "MR" is the cautious choice; a global extinction is MV.
LEVEL3_CATEGORY = "MR"
# Use only records with an affected native taxon (EICAT guidelines: a
# change in an ecosystem property alone is not an EICAT impact).
NATIVE_TAXA_ONLY = True
# Directions that can give "MC" (level 0). "Neutral" = no change found.
MC_DIRECTIONS = ("Negative", "Neutral")
# SEICAT category per GIDIAS magnitude of a negative impact on good
# quality of life (same caution for level 3).
LEVEL_TO_SEICAT = {0: "MC", 1: "MN", 2: "MO", 3: "MR"}
# Evidence label: number of independent sources at the highest category.
CONFIDENCE_SOURCES = {"high": 3, "medium": 2}  # fewer: "low"
MAX_LISTED = 12  # longest list of countries kept per species
MAX_REFERENCES = 5  # sources listed in eicat_reference ...
MAX_REFERENCE_CHARS = 160  # ... each cut to this length

# ---------------------------------------------------------------- LLM scope
TOP_X = 100  # top species of every list that should have impact data
LOCATIONS = "all"  # or a list of location names as in the scan files
MAX_SPECIES = None  # e.g. 50: cap on species scored in this run (cost)
# Which gaps the model fills. "eicat": species without a GIDIAS EICAT
# category. "seicat": also species with a GIDIAS EICAT category but no
# GIDIAS SEICAT category (many more calls).
FILL_GAPS = ("eicat", "seicat")
# Earlier LLM answers in impact_llm_scores.csv:
#   "if_changed": score again when MODEL or PROMPT_VERSION differs
#   "never"     : keep them as they are
#   "always"    : score again (cached answers are still reused)
RESCORE_LLM = "if_changed"

# ---------------------------------------------------------------- the model
# "dry_run"          : made-up answers; shown, but NOT merged into the
#                      master file (which then holds GIDIAS only)
# "openai_compatible": any server with the /chat/completions API (local
#                      servers such as Ollama, vLLM, LM Studio or
#                      llama.cpp, and many hosted services)
# "custom"           : your own function, see "Plug in a model" below
PROVIDER = "dry_run"
MODEL = ""  # model name exactly as your server or provider calls it
API_BASE = "http://localhost:11434/v1"  # base URL of the server
API_KEY_ENV = "LLM_API_KEY"  # name of the environment variable with the key
TEMPERATURE = 0.2  # > 0 so that repeated runs can disagree
MAX_TOKENS = 1400
REQUEST_TIMEOUT = 120  # seconds
MAX_RETRIES = 3
SLEEP_BETWEEN_CALLS = 0.0  # seconds; raise it if the server limits requests
N_REPEATS = 3  # independent answers per species (odd numbers work best)
PROMPT_VERSION = "impact-v1"  # change it when you change the prompt
# Evidence given to the model with the question (never in the benchmark):
#   "none"         : the model answers from what it knows
#   "folder"       : text from EVIDENCE_DIR/<Genus_species>.txt
#   "gidias+folder": also the GIDIAS text excerpts of the species, for
#                    species GIDIAS has records of but could not rate
EVIDENCE = "gidias+folder"
MAX_EVIDENCE_CHARS = 6000

# ---------------------------------------------------------------- benchmark
BENCHMARK_N = 100  # GIDIAS-rated species scored for comparison (0 = skip)
BENCHMARK_SEED = 42

DRY_RUN = PROVIDER == "dry_run"
assert RESCORE_LLM in ("if_changed", "never", "always"), RESCORE_LLM
assert PROVIDER in ("dry_run", "openai_compatible", "custom"), PROVIDER
assert EVIDENCE in ("none", "folder", "gidias+folder"), EVIDENCE
assert N_REPEATS >= 1 and TOP_X >= 1
if not DRY_RUN and not MODEL:
    raise ValueError("Set MODEL to the name of the model you plugged in; it "
                     "is stored with every category it produces.")
MODEL_LABEL = "dry-run (not a model)" if DRY_RUN else MODEL

(DATA / "cache").mkdir(parents=True, exist_ok=True)
for f in (GIDIAS_CSV, TAXONOMY_CSV, NAME_AUDIT_CSV, EXPORT_DIR, EXPERT_CSV):
    print(f"[{'OK ' if f.exists() else 'MISSING'}] {f}")
if not GIDIAS_CSV.exists():
    raise FileNotFoundError(f"Put the GIDIAS file at {GIDIAS_CSV}")
if USE_GBIF_API and "@institution.org" in CONTACT_EMAIL:
    warnings.warn("Set CONTACT_EMAIL so GBIF can contact you if needed.")
print(f"provider: {PROVIDER} | model: {MODEL_LABEL}")
if DRY_RUN:
    print("DRY RUN: LLM answers are made up and are not merged; the "
          "master file will hold GIDIAS only.")

[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\GIDIAS_v1.1_20260723_machine_read.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\taxonomy_backbone.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\name_matching_audit.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\model_outputs\exports
[MISSING] c:\Users\simon\Documents\GitHub\horizon-scanner\data\eicat_lookup.csv
provider: dry_run | model: dry-run (not a model)
DRY RUN: LLM answers are made up and are not merged; the master file will hold GIDIAS only.


### Shared vocabulary

GIDIAS and the model use the same terms. Everything the model returns is
mapped onto these lists; a term outside them is dropped.

In [5]:
EICAT_ORDER = ["MC", "MN", "MO", "MR", "MV"]  # ordered by magnitude
VALID = EICAT_ORDER + ["DD"]
CONFIDENCE = ["low", "medium", "high"]
EICAT_LABEL = {
    "MC": "no reduction in the performance of native individuals",
    "MN": "reduced performance of native individuals",
    "MO": "decline of a native population",
    "MR": "local extinction of a native taxon",
    "MV": "global extinction of a native taxon",
}
SEICAT_LABEL = {
    "MC": "no change in people's activities",
    "MN": "activities made more difficult",
    "MO": "fewer people taking part in an activity",
    "MR": "local disappearance of an activity",
    "MV": "irreversible local disappearance of an activity",
}
# EICAT mechanisms as written in GIDIAS (numbers removed, lower case).
MECHANISMS = [
    "competition", "predation", "hybridisation", "transmission of disease",
    "parasitism", "poisoning/toxicity",
    "bio-fouling or other direct physical disturbance",
    "grazing/herbivory/browsing",
    "chemical or physical or structural impact on ecosystem",
    "indirect impacts through interactions with other species"]
TAXA = ["Plant", "Invertebrate", "Vertebrate", "Microbe"]
REALMS = ["Terrestrial", "Freshwater", "Marine"]
# Constituents of good quality of life (SEICAT / IPBES).
CONSTITUENTS = ["Safety", "Material and immaterial assets", "Health",
                "Social, spiritual and cultural relations",
                "Freedom of choice and action"]
# IPBES categories of nature's contributions to people.
NCP = [
    "Habitat creation and maintenance",
    "Pollination and dispersal of seeds", "Regulation of air quality",
    "Regulation of climate", "Regulation of ocean acidification",
    "Regulation of freshwater quantity, location and timing",
    "Regulation of freshwater and coastal water quality",
    "Formation, protection and decontamination of soils and sediments",
    "Regulation of hazards and extreme events",
    "Regulation of detrimental organisms and biological processes",
    "Energy", "Food and feed", "Materials, companionship and labor",
    "Medicinal, biochemical and genetic resources",
    "Learning and inspiration", "Physical and psychological experiences",
    "Supporting identities", "Maintenance of options"]

### Plug in a model

The LLM part needs exactly one thing from a model:

```python
ask_model(system: str, user: str) -> str   # returns the model's text answer
```

| `PROVIDER` | What to set | Notes |
|---|---|---|
| `"openai_compatible"` | `API_BASE`, `MODEL`, and the key in the environment variable named by `API_KEY_ENV` (not needed for most local servers) | Works with every server that offers the `/chat/completions` interface. Set the key **before** starting Jupyter (`export LLM_API_KEY=...`); never write it in the notebook. |
| `"custom"` | write the body of `custom_model` below | For any other provider or a model loaded in Python. Set `MODEL` to a name that identifies it. |
| `"dry_run"` | nothing | Made-up answers to test the notebook; not merged. |

**Choosing a model.** The benchmark in Step 9 is the test. A model that
often disagrees with GIDIAS, or that rarely answers "data deficient" for
obscure species, should not be used. Models that can search the
literature are preferable to models that answer from memory.

In [6]:
def openai_compatible_model(system, user):
    """One chat completion from a server with the OpenAI-style API.

    Sends ``system`` and ``user`` to ``API_BASE/chat/completions`` and
    returns the text of the first choice. Rate limits (429) and server
    errors (5xx) are retried ``MAX_RETRIES`` times with a growing pause.
    """
    body = json.dumps({
        "model": MODEL, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}]}).encode()
    headers = {"Content-Type": "application/json"}
    key = os.environ.get(API_KEY_ENV)
    if key:
        headers["Authorization"] = f"Bearer {key}"
    url = API_BASE.rstrip("/") + "/chat/completions"
    last = None
    for attempt in range(MAX_RETRIES):
        request = urllib.request.Request(url, data=body, headers=headers,
                                         method="POST")
        try:
            with urllib.request.urlopen(request,
                                        timeout=REQUEST_TIMEOUT) as resp:
                data = json.load(resp)
            return data["choices"][0]["message"]["content"]
        except urllib.error.HTTPError as err:
            detail = err.read().decode(errors="ignore")[:300]
            last = f"HTTP {err.code}: {detail}"
            if err.code not in (408, 409, 429) and err.code < 500:
                break  # a wrong request does not get better by retrying
        except (urllib.error.URLError, TimeoutError, KeyError, IndexError,
                json.JSONDecodeError) as err:
            last = f"{type(err).__name__}: {err}"
        time.sleep(2 ** attempt)
    raise RuntimeError(f"The model server did not answer ({last}).")


def custom_model(system, user):
    """Your own model: return its text answer to ``system`` + ``user``.

    Replace the body, for example with a call to your provider's Python
    client or to a model loaded in this session.
    """
    raise NotImplementedError("Write custom_model() or choose another "
                              "PROVIDER.")


def dry_run_model(system, user):
    """Stand-in for a model: a made-up, repeatable answer per species.

    Only for testing the notebook. The categories depend on the species
    name alone and carry no information.
    """
    name = re.search(r"Species: (.+)", user).group(1).strip()
    pick = int(hashlib.md5(name.encode()).hexdigest(), 16)
    eicat = (VALID * 2 + ["DD"])[pick % 13]
    seicat = (VALID + ["DD", "DD"])[(pick // 13) % 8]
    return json.dumps({
        "knows_species": eicat != "DD",
        "eicat": {"category": eicat, "confidence": CONFIDENCE[pick % 3],
                  "mechanisms": [MECHANISMS[pick % 10]],
                  "affected_native_taxa": [TAXA[pick % 4]],
                  "justification": "Made-up answer of the dry run."},
        "seicat": {"category": seicat, "confidence": "low",
                   "constituents": [CONSTITUENTS[pick % 5]],
                   "justification": "Made-up answer of the dry run."},
        "ncp_negative": [NCP[pick % 18]], "realms": [REALMS[pick % 3]],
        "countries": ["Nowhere"], "positive_nature": pick % 7 == 0,
        "positive_people": pick % 5 == 0, "evidence": []})


ask_model = {"openai_compatible": openai_compatible_model,
             "custom": custom_model, "dry_run": dry_run_model}[PROVIDER]

### Helper functions

(a) text helpers, (b) species-key matching, (c) the GIDIAS summary per
species. The LLM functions follow in Step 6, next to their explanation.
Nothing is computed until Step 1.

In [7]:
# ======================================================================
# (a) Text helpers
# ======================================================================
MISSING = {"", "na", "nan", "none", "_", "not specified"}


def clean_text(value):
    """Trimmed single-spaced string, or None for an empty / NA cell."""
    if not isinstance(value, str):
        return None
    out = re.sub(r"\s+", " ", value).strip().strip('"').strip()
    return None if out.lower() in MISSING else out


def canonical_binomial(name):
    """'Acer negundo L.' -> 'acer negundo'; None if not a species name.

    Authorship, infraspecific parts and hybrid signs are dropped, so
    subspecies and varieties fall on their species (as in notebook 03
    and ``horizon_core.canonical_binomial``).
    """
    if not isinstance(name, str):
        return None
    words = name.replace("×", " ").split()
    if len(words) < 2:
        return None
    genus, epithet = words[0], words[1]
    if not (genus[:1].isupper() and genus.isalpha()):
        return None
    if not (epithet.replace("-", "").isalpha() and epithet.islower()
            and len(epithet) > 1):
        return None
    return f"{genus.lower()} {epithet}"


def split_list(value, sep=";"):
    """'Spain; Portugal' -> ['Spain', 'Portugal'] (cleaned, no blanks)."""
    if not isinstance(value, str):
        return []
    parts = [re.sub(r"[^\w\s().,'/-]", "", p).strip()
             for p in value.split(sep)]
    return [p for p in parts if p and p.lower() not in MISSING]


def clean_mechanism(value):
    """'(1) Competition; (2) Predation' -> ['competition', 'predation'].

    The leading EICAT mechanism numbers are removed. GIDIAS marks
    beneficial mechanisms with '(positive impact)'; that note is kept.
    """
    out = []
    for part in split_list(value):
        note = re.match(r"^\(([^)]*)\)\s*", part)
        text = part[note.end():] if note else part
        text = text.strip().lower()
        if note and "positive" in note.group(1).lower():
            text += " (positive)"
        if text:
            out.append(text)
    return out


def join_unique(values, limit=None, sep="; "):
    """Sorted unique non-empty strings joined; '+N more' past ``limit``."""
    items = sorted({v for v in values if isinstance(v, str) and v})
    if limit and len(items) > limit:
        items = items[:limit] + [f"+{len(items) - limit} more"]
    return sep.join(items)


def join_by_count(lists, limit=None, sep="; "):
    """Items of several lists, most frequent first, joined."""
    counts = Counter(item for items in lists for item in items)
    items = [k for k, _ in sorted(counts.items(),
                                  key=lambda kv: (-kv[1], kv[0]))]
    if limit and len(items) > limit:
        items = items[:limit] + [f"+{len(items) - limit} more"]
    return sep.join(items)


def backup(path):
    """Rename an existing output to *_previous.* before overwriting it."""
    if BACKUP_PREVIOUS and path.exists():
        path.replace(path.with_name(path.stem + "_previous" + path.suffix))

In [8]:
# ======================================================================
# (b) Species keys: tables of notebook 03 first, then the GBIF API
# ======================================================================
MATCH_FIELDS = [  # same layout as the cache of notebook 03
    "usageKey", "acceptedUsageKey", "scientificName", "canonicalName",
    "rank", "status", "matchType", "confidence",
    "kingdom", "phylum", "class", "order", "family", "genus", "species",
    "kingdomKey", "phylumKey", "classKey", "orderKey", "familyKey",
    "genusKey", "speciesKey",
]


def build_local_index(taxonomy_csv, audit_csv):
    """{canonical binomial: {gbif_id: kingdom}} from notebook 03.

    Accepted species names come from ``taxonomy_backbone.csv``; every
    SInAS / GloBI name that was matched to a species (synonyms included)
    comes from ``name_matching_audit.csv``. A missing file is skipped.
    """
    index = {}

    def add(names, keys, kingdoms):
        for name, key, kingdom in zip(names, keys, kingdoms):
            binomial = canonical_binomial(name)
            if binomial and pd.notna(key):
                kingdom = kingdom if isinstance(kingdom, str) else ""
                index.setdefault(binomial, {})[int(key)] = kingdom.lower()

    if Path(taxonomy_csv).exists():
        tax = pd.read_csv(taxonomy_csv, low_memory=False,
                          usecols=["gbif_id", "rank", "canonical_name",
                                   "kingdom"])
        tax = tax[tax["rank"].astype(str).str.upper() == "SPECIES"]
        add(tax["canonical_name"], tax["gbif_id"], tax["kingdom"])
    if Path(audit_csv).exists():
        cols = pd.read_csv(audit_csv, nrows=0).columns
        use = [c for c in ("name", "canonicalName", "species_key",
                           "kingdom") if c in cols]
        if "species_key" in use:
            audit = pd.read_csv(audit_csv, usecols=use, low_memory=False)
            audit = audit.dropna(subset=["species_key"])
            kingdom = (audit["kingdom"] if "kingdom" in audit
                       else pd.Series("", index=audit.index))
            for col in ("name", "canonicalName"):
                if col in audit:
                    add(audit[col], audit["species_key"], kingdom)
    return index


def match_local(name, kingdom, index):
    """Species key of ``name`` from the local index, or None.

    A name used in two kingdoms is settled by ``kingdom``; if that does
    not leave exactly one key, the name is left to the GBIF matcher.
    """
    hits = index.get(canonical_binomial(name))
    if not hits:
        return None
    if len(hits) > 1 and isinstance(kingdom, str):
        same = {k: v for k, v in hits.items() if v == kingdom.lower()}
        hits = same or hits
    return next(iter(hits)) if len(hits) == 1 else None


def load_jsonl_cache(path):
    """Read a JSONL cache into {key: record}; broken lines are skipped."""
    cache = {}
    if Path(path).exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                cache[rec["_key"]] = rec
    return cache


def gbif_match_one(name, kingdom):
    """One name-match request with retries. Returns a record or None.

    None means a transient failure (time-out, 429, 5xx); it is not
    cached, so a re-run tries again.
    """
    params = {"name": name, "strict": "false", "verbose": "false"}
    if kingdom:
        params["kingdom"] = kingdom
    headers = {"User-Agent": f"IAS-horizon-scanner ({CONTACT_EMAIL})"}
    for attempt in range(GBIF_RETRIES):
        try:
            resp = requests.get(GBIF_MATCH_URL, params=params,
                                headers=headers, timeout=GBIF_TIMEOUT)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        data = resp.json() if resp.status_code == 200 else {}
        return {f: data.get(f) for f in MATCH_FIELDS}
    return None


def match_gbif(pairs, cache_path):
    """Match (name, kingdom) pairs with the GBIF API, using the cache.

    The cache file and its keys are those of notebook 03, so names
    already asked there are not asked again.

    Returns
    -------
    dict
        {(name, kingdom): record in MATCH_FIELDS layout}
    """
    cache = load_jsonl_cache(cache_path)
    key_of = {p: f"{p[0]}|{p[1] or ''}" for p in pairs}
    todo = [p for p in pairs if key_of[p] not in cache]
    print(f"   GBIF: {len(pairs):,} names | {len(pairs) - len(todo):,} "
          f"cached | {len(todo):,} to query")
    failed = 0
    if todo:
        with ThreadPoolExecutor(MAX_CONCURRENT) as pool:
            answers = list(pool.map(lambda p: gbif_match_one(*p), todo))
        with open(cache_path, "a", encoding="utf-8") as fh:
            for pair, rec in zip(todo, answers):
                if rec is None:
                    failed += 1
                    continue
                rec["_key"] = key_of[pair]
                cache[key_of[pair]] = rec
                fh.write(json.dumps(rec) + "\n")
    if failed:
        warnings.warn(f"GBIF: {failed:,} requests failed (network); "
                      "re-run this cell to retry them.")
    return {p: cache[key_of[p]] for p in pairs if key_of[p] in cache}


def species_key_of(rec):
    """(species key or None, flag) for one GBIF match record.

    Rules of notebook 03: EXACT, or FUZZY with enough confidence, at
    species rank or below; the species key rolls up infraspecific names
    and synonyms.
    """
    match_type = rec.get("matchType") or "NONE"
    confidence = rec.get("confidence") or 0
    rank = rec.get("rank") or ""
    if match_type == "NONE":
        return None, "no_match"
    if match_type == "HIGHERRANK" or rank not in SPECIES_RANKS:
        return None, "higherrank"
    if match_type == "FUZZY" and confidence < MIN_FUZZY_CONFIDENCE:
        return None, "fuzzy_low_confidence"
    key = rec.get("speciesKey")
    if key is None and rank == "SPECIES":
        synonym = rec.get("status") == "SYNONYM"
        key = ((rec.get("acceptedUsageKey") if synonym else None)
               or rec.get("usageKey"))
    if key is None:
        return None, "no_species_key"
    return int(key), "fuzzy" if match_type == "FUZZY" else "exact"

In [9]:
# ======================================================================
# (c) Per-species summary
# ======================================================================
def n_sources(frame):
    """Number of distinct sources (DOI, else the reference text)."""
    return int(frame["source_id"].nunique())


def confidence_label(n):
    """Evidence label from the number of sources at the top category."""
    if n >= CONFIDENCE_SOURCES["high"]:
        return "high"
    return "medium" if n >= CONFIDENCE_SOURCES["medium"] else "low"


def plural(n, word, many=None):
    """'1 record', '3 records', '2 countries'."""
    return f"{n:,} {word if n == 1 else (many or word + 's')}"


def summarise_species(grp):
    """All per-species impact fields from the records of one species.

    Parameters
    ----------
    grp : pandas.DataFrame
        Rows of ``records`` of one ``gbif_id``.

    Returns
    -------
    dict
        The GIDIAS columns of one master-file row.
    """
    found = grp[grp["evidence_found"]]
    rated = found[found["eicat_record"] != ""]
    row = {
        "taxon_name": grp["taxon_name"].mode().iat[0],
        "gidias_names": join_unique(grp["gidias_name"]),
        "kingdom": grp["kingdom"].dropna().mode().get(0, ""),
        "ias_group": grp["ias_group"].dropna().mode().get(0, ""),
        "impact_n_records": int(len(found)),
        "impact_n_sources": n_sources(found),
        "impact_year_min": found["year"].min(),
        "impact_year_max": found["year"].max(),
    }
    # ---- EICAT: highest category over all rated records
    if len(rated):
        top = max(rated["eicat_record"], key=EICAT_ORDER.index)
        at_top = rated[rated["eicat_record"] == top]
        srcs = n_sources(at_top)
        places = sorted({c for cs in at_top["countries"] for c in cs})
        text = (f"{GIDIAS_VERSION}: highest documented negative impact on "
                f"native species is {EICAT_LABEL[top]} "
                f"({plural(len(at_top), 'record')} from "
                f"{plural(srcs, 'source')}"
                + (f" in {plural(len(places), 'country', 'countries')}"
                   if places else "") + ").")
        if top == "MR" and LEVEL3_CATEGORY == "MR":
            text += (" GIDIAS does not record whether the extinction is "
                     "reversible, so MV cannot be ruled out.")
        row.update(
            eicat=top, eicat_confidence=confidence_label(srcs),
            eicat_mechanism=join_by_count(at_top["mechanisms"]),
            eicat_n_records=int(len(at_top)), eicat_n_sources=srcs,
            eicat_countries=join_unique(places, MAX_LISTED),
            eicat_reference=join_unique(
                at_top["source_id"].str.slice(0, MAX_REFERENCE_CHARS),
                MAX_REFERENCES, sep=" | "))
    else:
        text = (f"{GIDIAS_VERSION}: "
                + (f"{plural(len(found), 'impact record')}, none with a "
                   "magnitude for a negative impact on native species."
                   if len(found) else
                   "the species was checked and no documented impact was "
                   "found."))
        row.update(eicat="DD", eicat_confidence="", eicat_mechanism="",
                   eicat_n_records=0, eicat_n_sources=0,
                   eicat_countries="", eicat_reference="")
    row["eicat_justification"] = (text + " Evidence summary, not an IUCN "
                                  "EICAT assessment.")
    row["eicat_agreement"] = np.nan  # LLM rows only
    row["seicat_agreement"] = np.nan
    row["eicat_records_by_category"] = "; ".join(
        f"{c}: {n}" for c in EICAT_ORDER
        if (n := int((rated["eicat_record"] == c).sum())))
    row["global_extinction"] = bool(found["global_extinction"].any())
    # ---- nature, all directions
    for direction in ("negative", "neutral", "positive"):
        row[f"nature_n_{direction}"] = int(
            (found["nature_direction"] == direction.capitalize()).sum())
    row["nature_positive_max_level"] = found["positive_level"].max()
    row["affected_native_taxa"] = join_unique(found["affected_taxon"])
    row["nature_mechanisms"] = join_by_count(found["mechanisms"])
    # ---- good quality of life (SEICAT) and nature's contributions
    rated_s = found[found["seicat_record"] != ""]
    if len(rated_s):
        top_s = max(rated_s["seicat_record"], key=EICAT_ORDER.index)
        at_top_s = rated_s[rated_s["seicat_record"] == top_s]
        srcs_s = n_sources(at_top_s)
        row.update(
            seicat=top_s, seicat_confidence=confidence_label(srcs_s),
            seicat_n_records=int(len(at_top_s)), seicat_n_sources=srcs_s,
            seicat_constituents=join_by_count(at_top_s["cwb_constituents"]),
            seicat_justification=(
                f"{GIDIAS_VERSION}: highest documented negative impact on "
                f"good quality of life is {SEICAT_LABEL[top_s]} "
                f"({plural(len(at_top_s), 'record')} from "
                f"{plural(srcs_s, 'source')}). Evidence summary, not a "
                "SEICAT assessment."))
    else:
        row.update(seicat="", seicat_confidence="", seicat_n_records=0,
                   seicat_n_sources=0, seicat_constituents="",
                   seicat_justification="")
    row["cwb_n_negative"] = int((found["cwb_direction"] == "Negative").sum())
    row["cwb_n_positive"] = int((found["cwb_direction"] == "Positive").sum())
    row["cwb_constituents"] = join_by_count(found["cwb_constituents"])
    row["ncp_n_negative"] = int((found["ncp_direction"] == "Negative").sum())
    row["ncp_n_positive"] = int((found["ncp_direction"] == "Positive").sum())
    ncp_neg = found["ncp_direction"] == "Negative"
    row["ncp_categories"] = join_by_count(
        found.loc[ncp_neg, "ncp_categories"])
    row["ncp_categories_positive"] = join_by_count(
        found.loc[found["ncp_direction"] == "Positive", "ncp_categories"])
    row["positive_nature"] = row["nature_n_positive"] > 0
    row["positive_people"] = (row["cwb_n_positive"]
                              + row["ncp_n_positive"]) > 0
    # ---- where
    places = sorted({c for cs in found["countries"] for c in cs})
    row["impact_realms"] = join_unique(found["realm"])
    row["impact_n_countries"] = len(places)
    row["impact_countries"] = join_unique(places, MAX_LISTED)
    row["impact_iso3"] = join_unique(
        [c for cs in found["iso3"] for c in cs], sep=";")
    row["impact_units_of_analysis"] = join_by_count(
        found["units_of_analysis"], 6)
    row["impact_on_islands"] = bool(found["island"].any())
    row["impact_in_protected_areas"] = bool(found["protected_area"].any())
    row["native_range_gidias"] = join_unique(found["native_range"], 4)
    return row

## Step 1 · Read GIDIAS and tidy the fields

1. Read the csv (all 94 columns, about 22 900 rows).
2. Species name: `Verified.Name.GBIF.Taxon` (the name the GIDIAS authors
   checked against GBIF); rows without it use `IAS.Species.Name`. Rows
   that are not a taxon (`[miscellaneous]`) are dropped.
3. Countries (`Country.Location`, `;`-separated) and mechanisms are split
   into lists. If `country_converter` is installed, ISO3 codes are added.
4. A source is identified by its DOI, or by the reference text when
   there is no DOI (72 % of the rows).

Rows with `checklist.used = No_evidence_found` mean "species checked, no
documented impact found"; they carry no impact but show the species was
looked at.

In [10]:
print("Step 1: GIDIAS")
raw = pd.read_csv(GIDIAS_CSV, low_memory=False, encoding="utf-8-sig")
required = ["Verified.Name.GBIF.Taxon", "IAS.Species.Name", "Kingdom",
            "checklist.used", "Reference", "Country.Location",
            "direction.Nature", "direction.Nature.Taxononly",
            "magnitude.Nature", "global.extinction",
            "mechanism.Nature.clean", "direction.CWB", "magnitude.CWB"]
missing_cols = [c for c in required if c not in raw.columns]
if missing_cols:
    raise ValueError(f"GIDIAS file lacks columns {missing_cols}; this "
                     "notebook was written for the v1.1 machine-read file.")

g = pd.DataFrame({
    "gidias_row_id": raw["rowID"],
    "gidias_unique_id": raw["UniqueID"],
    "gidias_name": raw["Verified.Name.GBIF.Taxon"].map(clean_text),
    "kingdom": raw["Kingdom"].map(clean_text),
    "ias_group": raw["IAS.Taxon"].map(clean_text),
    "evidence_found": raw["checklist.used"].eq("Evidence_found"),
})
original = raw["IAS.Species.Name"].map(clean_text)
g["gidias_name"] = g["gidias_name"].fillna(original)
g["gidias_name_original"] = original
g["reference"] = raw["Reference"].map(clean_text)
g["doi"] = (raw["DOI"].map(clean_text).str.rstrip(".")
            .str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True))
g["text_excerpt"] = raw["Text.excerpt"].map(clean_text)
g["source_id"] = g["doi"].fillna(g["reference"]).fillna("")
g.loc[~g["evidence_found"], "source_id"] = ""
g["year"] = pd.to_numeric(raw["Year"], errors="coerce").astype("Int64")
g["peer_reviewed"] = raw["Peer.reviewed"].astype(bool)
g["source_type"] = raw["Type.of.source"].map(clean_text)
g["region"] = raw["Region"].map(clean_text)
g["countries"] = raw["Country.Location"].map(split_list)
g["realm"] = raw["Realm"].map(clean_text)
g["units_of_analysis"] = raw["Units.of.Analysis.clean"].map(split_list)
g["spatial_scale"] = raw["Spatial.scale"].map(clean_text)
g["island"] = raw["Island.k"].astype(bool)
g["protected_area"] = raw["Protected.area.k"].astype(bool)
g["native_range"] = raw["native.range.of.IAS"].map(clean_text)
g["affected_taxon"] = raw["Affected.native.species.Taxon"].map(clean_text)
g["mechanisms"] = raw["mechanism.Nature.clean"].map(clean_mechanism)
g["nature_direction"] = raw["direction.Nature"].map(clean_text)
g["nature_direction_taxa"] = raw["direction.Nature.Taxononly"].map(
    clean_text)
g["nature_level"] = pd.to_numeric(raw["magnitude.Nature"],
                                  errors="coerce").astype("Int64")
g["global_extinction"] = raw["global.extinction"].astype(bool)
g["ncp_categories"] = raw["affected.NCP.clean"].map(
    lambda v: [re.sub(r"^\d+\.\s*", "", p) for p in split_list(v)])
g["ncp_direction"] = raw["direction.NCP"].map(clean_text)
g["cwb_constituents"] = raw["affected.CWB.clean"].map(split_list)
g["cwb_direction"] = raw["direction.CWB"].map(clean_text)
g["cwb_level"] = pd.to_numeric(raw["magnitude.CWB"],
                               errors="coerce").astype("Int64")

is_taxon = g["gidias_name"].notna() & ~g["gidias_name"].fillna(
    "").str.startswith("[")
print(f"   {len(g):,} rows | dropped {int((~is_taxon).sum())} rows that "
      "are not a taxon")
g = g[is_taxon].reset_index(drop=True)

# --- ISO3 codes of the countries (optional)
try:
    import country_converter as coco
    country_names = sorted({c for cs in g["countries"] for c in cs})
    codes = coco.convert(names=country_names, to="ISO3", not_found=None)
    codes = [codes] if isinstance(codes, str) else codes
    iso_of = {n: c for n, c in zip(country_names, codes)
              if isinstance(c, str) and len(c) == 3}
    print(f"   ISO3 codes for {len(iso_of)} of {len(country_names)} "
          "country names")
except ImportError:
    iso_of = {}
    print("   country_converter not installed: impact_iso3 stays empty")
g["iso3"] = g["countries"].map(
    lambda cs: sorted({iso_of[c] for c in cs if c in iso_of}))

print(f"   {g['gidias_name'].nunique():,} names | "
      f"{int(g['evidence_found'].sum()):,} impact records | "
      f"{int((~g['evidence_found']).sum()):,} 'no evidence found' rows")
print(g["ias_group"].value_counts().to_string())

Step 1: GIDIAS


18 European countries not found in regex
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More than one regular expression match for Bermuda (United Kingdom)
More th

   22,865 rows | dropped 7 rows that are not a taxon


More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular expression match for Curacao (Netherlands)
More than one regular exp

   ISO3 codes for 219 of 252 country names
   3,384 names | 22,243 impact records | 615 'no evidence found' rows
ias_group
Plant           9583
Invertebrate    7686
Vertebrate      4998
Microbe          591


## Step 2 · Match species names to GBIF Backbone species keys

The lists use the GBIF Backbone **species** key (`gbif_id`). Each GIDIAS
name is resolved in two stages:

| Stage | Source | `match_source` |
|---|---|---|
| 1 | Names notebook 03 already matched (accepted names and synonyms of the SInAS and GloBI species), compared as genus + epithet | `notebook03` |
| 2 | GBIF name-match API with the GIDIAS kingdom as a hint, same acceptance rules as notebook 03 | `gbif_api` |

Stage 1 guarantees that a species of the model gets exactly the key the
model uses. Names that stay unmatched are mostly viruses, genus-level
entries and informal names; they are listed in
`gidias_name_matching_audit.csv` with their number of records.

In [11]:
print("Step 2: species keys")
names = (g.groupby(["gidias_name", "kingdom"], dropna=False)
         .size().rename("n_records").reset_index())
local_index = build_local_index(TAXONOMY_CSV, NAME_AUDIT_CSV)
print(f"   local index: {len(local_index):,} names from notebook 03")
names["gbif_id"] = [match_local(n, k, local_index)
                    for n, k in zip(names["gidias_name"], names["kingdom"])]
names["match_source"] = np.where(names["gbif_id"].notna(), "notebook03",
                                 "unmatched")
names["match_flag"] = np.where(names["gbif_id"].notna(), "local", "")
names["gbif_name"] = None

open_rows = names["gbif_id"].isna()
if USE_GBIF_API and open_rows.any():
    pairs = [(n, k if isinstance(k, str) else None) for n, k in
             zip(names.loc[open_rows, "gidias_name"],
                 names.loc[open_rows, "kingdom"])]
    answers = match_gbif(list(dict.fromkeys(pairs)), NAME_CACHE)
    for idx, pair in zip(names.index[open_rows], pairs):
        rec = answers.get(pair)
        if rec is None:
            names.at[idx, "match_flag"] = "request_failed"
            continue
        key, flag = species_key_of(rec)
        names.at[idx, "match_flag"] = flag
        names.at[idx, "gbif_name"] = rec.get("species") or rec.get(
            "canonicalName")
        if key is not None:
            names.at[idx, "gbif_id"] = key
            names.at[idx, "match_source"] = "gbif_api"
names["gbif_id"] = names["gbif_id"].astype("Int64")

matched = names["gbif_id"].notna()
print(names.groupby(["match_source", "match_flag"])["n_records"]
      .agg(["size", "sum"]).rename(columns={"size": "names",
                                            "sum": "records"}).to_string())
print(f"   matched: {matched.mean():.1%} of names, "
      f"{names.loc[matched, 'n_records'].sum() / len(g):.1%} of records "
      f"-> {names['gbif_id'].nunique():,} species keys")
print("   most frequent unmatched names:")
print(names[~matched].nlargest(10, "n_records")[
    ["gidias_name", "kingdom", "match_flag", "n_records"]].to_string(
    index=False))

Step 2: species keys
   local index: 376,243 names from notebook 03
   GBIF: 481 names | 139 cached | 342 to query
                                   names  records
match_source match_flag                          
gbif_api     exact                   290     1852
             fuzzy                     7        7
notebook03   local                  2903    20122
unmatched    fuzzy_low_confidence      3        3
             higherrank              180      873
             no_match                  1        1
   matched: 94.6% of names, 96.2% of records -> 3,187 species keys
   most frequent unmatched names:
    gidias_name  kingdom match_flag  n_records
Carpobrotus spp  Plantae higherrank        158
    Tamarix spp  Plantae higherrank         60
      Pinus spp  Plantae higherrank         54
  Dreissena spp Animalia higherrank         42
   Fallopia spp  Plantae higherrank         31
   Prosopis spp  Plantae higherrank         31
 Eucalyptus spp  Plantae higherrank         26
     Rat

## Step 3 · Code every record

**Impact on nature → EICAT.** GIDIAS gives each record a direction and a
magnitude that follow the EICAT levels:

| `magnitude.Nature` | Meaning | Category of the record |
|---|---|---|
| 0 | studied, no reduction in performance | MC (direction negative or neutral) |
| 1 | performance of native individuals reduced | MN |
| 2 | native population decline | MO |
| 3 | local extinction | MR (`LEVEL3_CATEGORY`) |
| any, `global.extinction = TRUE` | a native taxon went extinct | MV |

A record gets a category only if it found evidence, reports a
**negative** direction (level 1–3) and, with `NATIVE_TAXA_ONLY`, names an
affected native taxon (`direction.Nature.Taxononly`). Records with a
positive direction are kept as `positive_level` (the idea of EICAT+), not
mixed into the category.

**Impact on good quality of life → SEICAT.** `magnitude.CWB` (0–3) of
records with a negative direction, coded the same way.

**Nature's contributions to people (NCP).** `magnitude.NCP` is free text
in GIDIAS, so only the direction and the NCP categories are used.

In [12]:
print("Step 3: record categories")
records = g.merge(names[["gidias_name", "kingdom", "gbif_id",
                         "match_source"]],
                  on=["gidias_name", "kingdom"], how="left")

direction = (records["nature_direction_taxa"] if NATIVE_TAXA_ONLY
             else records["nature_direction"])
level = records["nature_level"]
usable = records["evidence_found"] & level.notna()
negative = usable & direction.eq("Negative")

eicat_record = pd.Series("", index=records.index)
for lvl, cat in LEVEL_TO_EICAT.items():
    ok = (usable & direction.isin(MC_DIRECTIONS) if cat == "MC"
          else negative)
    eicat_record[ok & level.eq(lvl)] = cat
eicat_record[negative & level.eq(3)] = LEVEL3_CATEGORY
eicat_record[records["evidence_found"] & records["global_extinction"]
             & direction.eq("Negative")] = "MV"
records["eicat_record"] = eicat_record

positive = usable & records["nature_direction"].eq("Positive")
records["positive_level"] = level.where(positive)

cwb_negative = (records["evidence_found"]
                & records["cwb_direction"].eq("Negative"))
records["seicat_record"] = (records["cwb_level"].map(LEVEL_TO_SEICAT)
                            .where(cwb_negative).fillna(""))

print("   EICAT category of the records:")
print(records["eicat_record"].replace("", "(none)").value_counts()
      .to_string())
print("   SEICAT category of the records:",
      records["seicat_record"].replace("", "(none)").value_counts()
      .to_dict())
print(f"   records with a positive impact on nature and a magnitude: "
      f"{int(positive.sum()):,}")

Step 3: record categories
   EICAT category of the records:
eicat_record
(none)    12854
MO         4203
MN         2999
MC         1593
MR         1177
MV           32
   SEICAT category of the records: {'(none)': 21526, 'MN': 781, 'MO': 400, 'MC': 80, 'MR': 71}
   records with a positive impact on nature and a magnitude: 669


## Step 4 · Summarise GIDIAS per species

One row per `gbif_id`. Several GIDIAS names can fall on one species
(synonyms, subspecies); their records are pooled.

* **`eicat`**, **`seicat`** — the highest category of any record.
  `eicat = DD` when GIDIAS has records for the species but none that can
  be given a category, or when the species was checked and nothing was
  found. `seicat` is empty when no record rates an impact on people.
* **`eicat_confidence`**, **`seicat_confidence`** — a label for the
  amount of evidence, **not** the EICAT confidence score: `high` from 3
  independent sources at the highest category, `medium` for 2, `low`
  for 1.
* Context: mechanisms, affected native taxa, realms, countries with
  documented impact, positive impacts, NCP categories and well-being
  constituents affected.

In [13]:
print("Step 4: GIDIAS per species")
with_key = records[records["gbif_id"].notna()].copy()
accepted = dict(zip(names["gbif_id"], names["gbif_name"]))
local_name = {}
if TAXONOMY_CSV.exists():
    tax = pd.read_csv(TAXONOMY_CSV, usecols=["gbif_id", "canonical_name"])
    local_name = dict(zip(tax["gbif_id"], tax["canonical_name"]))
with_key["taxon_name"] = [
    local_name.get(k) or accepted.get(k) or n
    for k, n in zip(with_key["gbif_id"], with_key["gidias_name"])]

rows = []
for gbif_id, grp in with_key.groupby("gbif_id", sort=True):
    rows.append({"gbif_id": int(gbif_id), **summarise_species(grp)})
gidias = pd.DataFrame(rows)
for col in ("impact_year_min", "impact_year_max",
            "nature_positive_max_level"):
    gidias[col] = gidias[col].astype("Int64")

print(f"   {len(gidias):,} species")
print("   EICAT: ", gidias["eicat"].value_counts()
      .reindex(VALID).fillna(0).astype(int).to_dict())
print("   SEICAT:", gidias["seicat"].replace("", "(none)")
      .value_counts().to_dict())
print("   evidence label of the EICAT categories MC-MV:",
      gidias.loc[gidias["eicat"] != "DD", "eicat_confidence"]
      .value_counts().to_dict())
gidias.sort_values("eicat_n_sources", ascending=False).head(8)[
    ["taxon_name", "eicat", "eicat_confidence", "eicat_n_sources",
     "seicat", "impact_n_records", "impact_n_countries", "impact_realms"]]

Step 4: GIDIAS per species
   3,187 species
   EICAT:  {'MC': 64, 'MN': 396, 'MO': 558, 'MR': 198, 'MV': 10, 'DD': 1961}
   SEICAT: {'(none)': 2809, 'MN': 176, 'MO': 126, 'MR': 40, 'MC': 36}
   evidence label of the EICAT categories MC-MV: {'low': 778, 'medium': 228, 'high': 220}


,taxon_name,eicat,eicat_confidence,eicat_n_sources,seicat,impact_n_records,impact_n_countries,impact_realms
208,Linepithema humile,MR,high,46,MO,194,12,Terrestrial
2200,Solenopsis invicta,MR,high,24,MR,324,9,Freshwater; Marine; Terrestrial
219,Anoplolepis gracilipes,MR,high,23,MR,118,13,Terrestrial
2967,Hymenoscyphus fraxineus,MO,high,20,MO,27,17,Terrestrial
1231,Acridotheres tristis,MO,high,16,MN,83,18,Terrestrial
243,Pheidole megacephala,MR,high,16,MN,57,17,Terrestrial
201,Wasmannia auropunctata,MR,high,15,MR,104,14,Terrestrial
1430,Pontederia crassipes,MR,high,15,MR,180,38,Freshwater; Terrestrial


## Step 5 · Read the lists and select the species GIDIAS does not cover

The lists say which species matter (their rank per location). A species
is selected for the model when it is in the top `TOP_X` of at least one
list **and**

* GIDIAS has no EICAT category MC–MV for it (`"eicat"` in `FILL_GAPS`),
  **or** GIDIAS has no SEICAT category for it (`"seicat"` in
  `FILL_GAPS`), **and**
* it has no LLM answer yet, or `RESCORE_LLM` asks for a new one.

One call returns both classifications; only the block GIDIAS lacks is
used. Species are ordered by their best rank, so with `MAX_SPECIES` the
most important ones come first. Without exported lists the LLM part is
skipped and the master file holds GIDIAS only.

In [14]:
SCAN_KEEP = ["gbif_id", "taxon_name", "target_country", "rank", "kingdom",
             "class", "family", "group"]
LLM_ID_COLUMNS = ["impact_model", "impact_prompt_version", "impact_date"]


def read_scans(export_dir, locations="all"):
    """All per-location scan files as one table (empty if there are none).

    The combined ``scan_all_locations.csv`` is skipped (it repeats the
    per-location files) unless it is the only scan file.
    """
    files = sorted(Path(export_dir).glob("scan_*.csv"))
    single = [f for f in files if f.name != "scan_all_locations.csv"]
    parts = []
    for f in single or files:
        df = pd.read_csv(f, low_memory=False)
        parts.append(df[[c for c in SCAN_KEEP if c in df.columns]])
    if not parts:
        return pd.DataFrame(columns=SCAN_KEEP)
    scans = pd.concat(parts, ignore_index=True)
    if locations != "all":
        scans = scans[scans["target_country"].isin(list(locations))]
    return scans


def read_llm_scores(path):
    """Earlier LLM answers (``impact_llm_scores.csv``), empty if none."""
    if not Path(path).exists():
        return pd.DataFrame(columns=["gbif_id"] + LLM_ID_COLUMNS)
    old = pd.read_csv(path, keep_default_na=False)
    old["gbif_id"] = old["gbif_id"].astype(np.int64)
    return old


print("Step 5: lists and species to score")
scans = read_scans(EXPORT_DIR, LOCATIONS)
old_llm = read_llm_scores(LLM_SCORES)
print(f"   {scans['target_country'].nunique()} location(s) | "
      f"{scans['gbif_id'].nunique():,} species in the lists | "
      f"{len(old_llm):,} earlier LLM answers")

g_eicat = dict(zip(gidias["gbif_id"], gidias["eicat"]))
g_seicat = dict(zip(gidias["gbif_id"], gidias["seicat"]))
top = scans[scans["rank"] <= TOP_X]
best = (top.sort_values("rank").drop_duplicates("gbif_id")
        .rename(columns={"rank": "best_rank",
                         "target_country": "best_rank_location"}))
best["n_lists"] = best["gbif_id"].map(
    top.groupby("gbif_id")["target_country"].nunique())
best["gidias_eicat"] = best["gbif_id"].map(g_eicat).fillna("")
best["gidias_seicat"] = best["gbif_id"].map(g_seicat).fillna("")
need = pd.Series(False, index=best.index)
if "eicat" in FILL_GAPS:
    need |= ~best["gidias_eicat"].isin(EICAT_ORDER)
if "seicat" in FILL_GAPS:
    need |= ~best["gidias_seicat"].isin(EICAT_ORDER)

have = old_llm.set_index("gbif_id")
has_llm = best["gbif_id"].isin(have.index)
outdated = has_llm & (
    (best["gbif_id"].map(have.get("impact_model", pd.Series(dtype=str)))
     != MODEL_LABEL)
    | (best["gbif_id"].map(have.get("impact_prompt_version",
                                    pd.Series(dtype=str)))
       != PROMPT_VERSION))
redo = {"never": has_llm & False, "if_changed": outdated,
        "always": has_llm}[RESCORE_LLM]
todo = best[need & (~has_llm | redo)].copy()
todo = todo.sort_values(["best_rank", "gbif_id"]).reset_index(drop=True)
if MAX_SPECIES:
    todo = todo.head(MAX_SPECIES)

print(f"   species in a top {TOP_X}: {len(best):,} | GIDIAS EICAT MC-MV: "
      f"{int(best['gidias_eicat'].isin(EICAT_ORDER).sum()):,} | GIDIAS "
      f"SEICAT: {int(best['gidias_seicat'].isin(EICAT_ORDER).sum()):,} | "
      f"with a gap: {int(need.sum()):,}")
print(f"   to score now: {len(todo):,} species x {N_REPEATS} answer(s) = "
      f"up to {len(todo) * N_REPEATS:,} model calls (cached answers are "
      "free)")
todo[[c for c in ("taxon_name", "group", "best_rank", "best_rank_location",
                  "n_lists", "gidias_eicat", "gidias_seicat")
      if c in todo.columns]].head(10)

Step 5: lists and species to score
   1 location(s) | 500 species in the lists | 0 earlier LLM answers
   species in a top 100: 100 | GIDIAS EICAT MC-MV: 6 | GIDIAS SEICAT: 3 | with a gap: 97
   to score now: 97 species x 3 answer(s) = up to 291 model calls (cached answers are free)


,taxon_name,group,best_rank,best_rank_location,n_lists,gidias_eicat,gidias_seicat
0,Trechus rubens,invertebrate,1,Belgium,1,,
1,Attagenus unicolor,invertebrate,2,Belgium,1,DD,
2,Ptinus tectus,invertebrate,3,Belgium,1,DD,
3,Anarsia lineatella,invertebrate,4,Belgium,1,,
4,Elatobium abietinum,invertebrate,5,Belgium,1,,
5,Epinotia fraternana,invertebrate,6,Belgium,1,,
6,Phytomyza gymnostoma,invertebrate,7,Belgium,1,,
7,Saturnia pyri,invertebrate,8,Belgium,1,,
8,Epinotia nigricana,invertebrate,9,Belgium,1,,
9,Trogoderma glabrum,invertebrate,10,Belgium,1,DD,


## Step 6 · Prompt and answer format (GIDIAS structure)

**The question.** The system text gives the EICAT and SEICAT definitions,
the shared vocabulary and the rules. The user text names the species
(with its taxonomy, so homonyms are not confused) and adds evidence, if
any.

| Risk with a language model | Rule in the prompt |
|---|---|
| Inventing impacts for a species it does not know | "If you do not recognise the species … answer DD" and a `knows_species` field |
| Rating potential instead of documented impact | "Only documented impacts in the alien range count" |
| Borrowing the reputation of a relative | "Do not infer from related species" |
| Mixing harm to nature and harm to people | two separate blocks, each with its own definition |
| Invented references | sources are asked for, marked "unverified" when unsure, and never shown as verified |

**The answer** is one JSON object with the fields GIDIAS has:

```json
{"knows_species": true,
 "eicat":  {"category": "MO", "confidence": "medium",
            "mechanisms": ["predation"],
            "affected_native_taxa": ["Vertebrate"], "justification": "…"},
 "seicat": {"category": "MN", "confidence": "low",
            "constituents": ["Material and immaterial assets"],
            "justification": "…"},
 "ncp_negative": ["Food and feed"], "realms": ["Freshwater"],
 "countries": ["Spain", "Italy"],
 "positive_nature": false, "positive_people": true,
 "evidence": [{"claim": "…", "where": "…", "source": "…"}]}
```

`parse_answer` checks it and maps every term onto the shared vocabulary.
An answer that cannot be read is not used. `knows_species = false`
always becomes DD in both blocks.

**Several answers per species** are combined per block: DD if at least
half of the answers are DD, otherwise the lower median of the rated
answers, which one outlying answer cannot move. `*_agreement` is the
share of answers equal to the final category; the confidence is the
lowest among the agreeing answers and never above "low" when the
agreement is below 2/3. List fields keep the terms named in at least
half of the answers.

**Changing the prompt?** Also change `PROMPT_VERSION`.

In [15]:
SYSTEM_PROMPT = f"""You assist an invasion-biology team that compiles \
DOCUMENTED impacts of alien species, following EICAT (IUCN Environmental \
Impact Classification for Alien Taxa) and SEICAT (Socio-Economic Impact \
Classification of Alien Taxa).

Task: for one species, report the highest magnitude of impact that is \
documented where the species is ALIEN (introduced), anywhere in the \
world, separately for native species (EICAT) and for people (SEICAT), \
with the context fields below.

EICAT categories (impact on NATIVE taxa):
- MC: negligible; no reduction in the performance of native individuals.
- MN: performance (fitness, growth, reproduction) of native individuals \
reduced, but no decline of native population sizes.
- MO: decline of the population size of at least one native taxon, but \
no local extinction.
- MR: local or sub-population extinction of at least one native taxon, \
naturally reversible if the alien is removed.
- MV: local extinction that is naturally irreversible (includes global \
extinction).
- DD: no adequate evidence to assign a category.

SEICAT categories (impact on human well-being, measured as change in \
people's ACTIVITIES; not monetary cost alone):
- MC: no deleterious impact reported; no change in activities.
- MN: activities are more difficult, but no fewer people take part.
- MO: fewer people take part in at least one activity.
- MR: an activity disappears from all or part of the invaded area, \
reversible.
- MV: an activity disappears irreversibly.
- DD: no adequate evidence to assign a category.

Vocabulary (use these exact terms):
- mechanisms: {"; ".join(MECHANISMS)}
- affected_native_taxa: {"; ".join(TAXA)}
- constituents (of good quality of life): {"; ".join(CONSTITUENTS)}
- ncp_negative (nature's contributions to people that are reduced): \
{"; ".join(NCP)}
- realms: {"; ".join(REALMS)}

Rules:
1. Only documented impacts in the alien range count. Do not rate \
potential or expected impact, or impacts in the native range.
2. Do not infer impacts from related species or from the species' traits.
3. If you do not recognise the species, are unsure which species is \
meant, or know of no documented impact evidence, answer DD in both \
blocks and set "knows_species" accordingly. DD is a correct and useful \
answer.
4. Rate EICAT and SEICAT independently. A species can be DD in one and \
rated in the other.
5. If evidence text is supplied, base the answer on it first.
6. Name sources only if you are confident they exist (author, year, \
journal or report). Write "unverified" as the source if you are not \
sure. Never invent a reference.
7. confidence: "high" = several consistent studies; "medium" = some \
direct evidence; "low" = little or indirect evidence.

Answer with ONE JSON object and nothing else:
{{"knows_species": true or false,
 "eicat": {{"category": "MC"|"MN"|"MO"|"MR"|"MV"|"DD",
  "confidence": "high"|"medium"|"low", "mechanisms": [],
  "affected_native_taxa": [],
  "justification": "at most 50 words: what impact, on which native \
taxa, where"}},
 "seicat": {{"category": "MC"|"MN"|"MO"|"MR"|"MV"|"DD",
  "confidence": "high"|"medium"|"low", "constituents": [],
  "justification": "at most 50 words: which activity, where"}},
 "ncp_negative": [], "realms": [],
 "countries": [countries with a documented impact, English names],
 "positive_nature": true or false (documented benefit to native species),
 "positive_people": true or false (documented benefit to people),
 "evidence": [{{"claim": "...", "where": "country or region", \
"source": "... or unverified"}}]}}"""


def build_prompt(species, evidence=""):
    """User message for one species.

    Parameters
    ----------
    species : mapping
        ``taxon_name`` and, if present, ``kingdom``, ``class``, ``family``.
    evidence : str
        Text given to the model as evidence ("" = none).
    """
    taxonomy = ", ".join(
        f"{rank}: {species[rank]}" for rank in ("kingdom", "class", "family")
        if isinstance(species.get(rank), str) and species.get(rank))
    text = f"Species: {species['taxon_name']}\n"
    if taxonomy:
        text += f"Taxonomy (GBIF Backbone): {taxonomy}\n"
    if evidence:
        text += ("\nEvidence supplied by the user (may be incomplete):\n"
                 f'"""\n{evidence}\n"""\n')
    return text + "\nReport the documented impacts as specified."


def to_vocabulary(values, vocabulary):
    """Terms of ``values`` that are in ``vocabulary`` (case-insensitive).

    A term that contains, or is contained in, exactly one vocabulary
    term is mapped onto it ("herbivory" -> "grazing/herbivory/browsing").
    """
    if isinstance(values, str):
        values = re.split(r"[;|]", values)
    out = []
    for value in values if isinstance(values, list) else []:
        term = " ".join(str(value).lower().split())
        if not term:
            continue
        exact = [v for v in vocabulary if v.lower() == term]
        loose = [v for v in vocabulary
                 if term in v.lower() or v.lower() in term]
        hit = exact or (loose if len(loose) == 1 else [])
        if hit and hit[0] not in out:
            out.append(hit[0])
    return out


def as_bool(value):
    """True for a JSON true or the words 'true' / 'yes'."""
    return value if isinstance(value, bool) else (
        str(value).strip().lower() in ("true", "yes"))


def parse_block(raw, list_key, vocabulary, knows):
    """One checked classification block (EICAT or SEICAT), or None."""
    raw = raw if isinstance(raw, dict) else {}
    category = str(raw.get("category", "")).strip().upper()
    if category not in VALID:
        return None
    confidence = str(raw.get("confidence", "")).strip().lower()
    confidence = confidence if confidence in CONFIDENCE else "low"
    if not knows:
        category, confidence = "DD", "low"
    rated = category != "DD"
    return {"category": category, "confidence": confidence,
            list_key: (to_vocabulary(raw.get(list_key), vocabulary)
                       if rated else []),
            "justification": " ".join(
                str(raw.get("justification", "")).split())[:500]}


def parse_answer(text):
    """Turn a model answer into a checked dict, or None if unusable.

    Accepts the JSON object alone, inside a code fence or surrounded by
    text. Both blocks must carry a valid category; every list is mapped
    onto the shared vocabulary; an unknown species becomes DD in both
    blocks.
    """
    if not isinstance(text, str):
        return None
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        raw = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    if not isinstance(raw, dict):
        return None
    knows = as_bool(raw.get("knows_species"))
    eicat = parse_block(raw.get("eicat"), "mechanisms", MECHANISMS, knows)
    seicat = parse_block(raw.get("seicat"), "constituents", CONSTITUENTS,
                         knows)
    if eicat is None or seicat is None:
        return None
    eicat["affected_native_taxa"] = (
        to_vocabulary((raw.get("eicat") or {}).get("affected_native_taxa"),
                      TAXA) if eicat["category"] != "DD" else [])
    any_rated = eicat["category"] != "DD" or seicat["category"] != "DD"
    countries = raw.get("countries") if any_rated else []
    countries = [" ".join(str(c).split())[:60] for c in
                 (countries if isinstance(countries, list) else [])
                 if str(c).strip()]
    evidence = raw.get("evidence")
    sources = [str(item["source"]).strip()
               for item in (evidence if isinstance(evidence, list) else [])
               if isinstance(item, dict)
               and str(item.get("source", "")).strip()]
    return {"knows_species": knows, "eicat": eicat, "seicat": seicat,
            "ncp_negative": (to_vocabulary(raw.get("ncp_negative"), NCP)
                             if any_rated else []),
            "realms": (to_vocabulary(raw.get("realms"), REALMS)
                       if any_rated else []),
            "countries": countries[:MAX_LISTED],
            "positive_nature": knows and as_bool(raw.get("positive_nature")),
            "positive_people": knows and as_bool(raw.get("positive_people")),
            "sources": sources}


def combine_block(blocks):
    """Final category, agreement, confidence and agreeing blocks.

    DD if at least half of the answers are DD; otherwise the (lower)
    median of the rated answers.
    """
    cats = [b["category"] for b in blocks]
    rated = sorted(EICAT_ORDER.index(c) for c in cats if c != "DD")
    if cats.count("DD") * 2 >= len(cats) or not rated:
        final = "DD"
    else:
        final = EICAT_ORDER[rated[(len(rated) - 1) // 2]]
    agree = [b for b in blocks if b["category"] == final]
    agreement = len(agree) / len(blocks)
    confidence = min((b["confidence"] for b in agree),
                     key=CONFIDENCE.index, default="low")
    if agreement < 2 / 3 or final == "DD":
        confidence = "low" if final != "DD" else ""
    return final, round(agreement, 2), confidence, agree


def majority(lists, n_answers):
    """Terms named in at least half of ``n_answers``, most frequent first."""
    counts = Counter(item for items in lists for item in dict.fromkeys(items))
    return [k for k, n in counts.most_common() if n * 2 >= n_answers]


def combine_answers(answers):
    """One master-file row (LLM columns) from several parsed answers.

    Returns None if ``answers`` is empty.
    """
    if not answers:
        return None
    n = len(answers)
    e_cat, e_agree, e_conf, e_blocks = combine_block(
        [a["eicat"] for a in answers])
    s_cat, s_agree, s_conf, s_blocks = combine_block(
        [a["seicat"] for a in answers])

    def longest(blocks):
        return max((b["justification"] for b in blocks), key=len,
                   default="")

    sources = list(dict.fromkeys(s for a in answers for s in a["sources"]))
    countries = majority([a["countries"] for a in answers], n)
    return {
        "eicat": e_cat, "eicat_confidence": e_conf,
        "eicat_agreement": e_agree,
        "eicat_mechanism": "; ".join(
            majority([b["mechanisms"] for b in e_blocks], len(e_blocks))),
        "affected_native_taxa": "; ".join(majority(
            [b["affected_native_taxa"] for b in e_blocks], len(e_blocks))),
        "eicat_justification": longest(e_blocks),
        "eicat_reference": " | ".join(sources)[:1000],
        "seicat": s_cat, "seicat_confidence": s_conf,
        "seicat_agreement": s_agree,
        "seicat_constituents": "; ".join(
            majority([b["constituents"] for b in s_blocks], len(s_blocks))),
        "seicat_justification": longest(s_blocks),
        "ncp_categories": "; ".join(
            majority([a["ncp_negative"] for a in answers], n)),
        "impact_realms": "; ".join(
            majority([a["realms"] for a in answers], n)),
        "impact_countries": "; ".join(countries),
        "impact_n_countries": len(countries),
        "positive_nature": sum(a["positive_nature"] for a in answers) * 2 > n,
        "positive_people": sum(a["positive_people"] for a in answers) * 2 > n,
        "impact_n_runs": n,
        "eicat_runs": "/".join(a["eicat"]["category"] for a in answers),
        "seicat_runs": "/".join(a["seicat"]["category"] for a in answers)}


def load_cache(path):
    """{key: raw answer} from the JSONL cache (broken lines skipped)."""
    cache = {}
    if Path(path).exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                cache[rec["key"]] = rec["answer"]
    return cache


def evidence_for(gbif_id, taxon_name):
    """Evidence text for a species ("" if none).

    Your own notes from EVIDENCE_DIR come first; with "gidias+folder"
    the GIDIAS text excerpts of the species follow, each with its
    reference, until MAX_EVIDENCE_CHARS is reached.
    """
    if EVIDENCE == "none":
        return ""
    parts = []
    path = EVIDENCE_DIR / (re.sub(r"\s+", "_", str(taxon_name).strip())
                           + ".txt")
    if path.exists():
        parts.append(path.read_text(encoding="utf-8", errors="ignore"))
    if EVIDENCE == "gidias+folder":
        parts += GIDIAS_EXCERPTS.get(int(gbif_id), [])
    return "\n".join(parts)[:MAX_EVIDENCE_CHARS].strip()


def score_species(table, purpose, use_evidence=True):
    """Ask the model about every species of ``table`` and combine.

    Each raw answer goes straight into the cache file, keyed by species,
    model, prompt version, run number and the evidence text.

    Parameters
    ----------
    table : pandas.DataFrame
        ``gbif_id``, ``taxon_name`` and optional taxonomy columns.
    purpose : str
        Progress-bar label.
    use_evidence : bool
        False for the benchmark, so GIDIAS text cannot leak into it.

    Returns
    -------
    (pandas.DataFrame, list)
        One row per species with a usable answer, and the gbif_ids that
        failed.
    """
    cache = load_cache(LLM_CACHE)
    rows, failed, n_calls = [], [], 0
    with open(LLM_CACHE, "a", encoding="utf-8") as fh:
        for species in tqdm(table.to_dict("records"), desc=purpose,
                            unit="sp"):
            evidence = (evidence_for(species["gbif_id"],
                                     species["taxon_name"])
                        if use_evidence else "")
            prompt = build_prompt(species, evidence)
            digest = hashlib.md5(evidence.encode()).hexdigest()[:8]
            answers = []
            for run in range(N_REPEATS):
                key = (f"{species['gbif_id']}|{MODEL_LABEL}|"
                       f"{PROMPT_VERSION}|{run}|{digest}")
                if key not in cache:
                    try:
                        cache[key] = ask_model(SYSTEM_PROMPT, prompt)
                    except NotImplementedError:
                        raise
                    except Exception as err:  # one failed call: go on
                        warnings.warn(f"{species['taxon_name']}: {err}")
                        continue
                    n_calls += 1
                    fh.write(json.dumps({"key": key, "answer": cache[key],
                                         "taxon_name": species["taxon_name"]})
                             + "\n")
                    fh.flush()
                    time.sleep(SLEEP_BETWEEN_CALLS)
                parsed = parse_answer(cache[key])
                if parsed is not None:
                    answers.append(parsed)
            combined = combine_answers(answers)
            if combined is None:
                failed.append(int(species["gbif_id"]))
                continue
            rows.append({"gbif_id": int(species["gbif_id"]),
                         "taxon_name": species["taxon_name"],
                         "kingdom": species.get("kingdom", ""), **combined,
                         "llm_has_evidence": bool(evidence)})
    print(f"   {purpose}: {len(rows):,} species scored, {len(failed):,} "
          f"failed, {n_calls:,} new model call(s)")
    return pd.DataFrame(rows), failed


# GIDIAS text excerpts per species, for the species GIDIAS has records of
# but could not rate (used as evidence; see EVIDENCE).
unrated = set(gidias.loc[gidias["eicat"] == "DD", "gbif_id"])
quotes = records[records["gbif_id"].isin(unrated) & records["evidence_found"]
                 & records["text_excerpt"].notna()]
GIDIAS_EXCERPTS = {
    int(k): [f"- {str(ref)[:120]}: {text}" for ref, text in
             zip(grp["reference"], grp["text_excerpt"])]
    for k, grp in quotes.groupby("gbif_id")}
print(f"GIDIAS excerpts available as evidence for "
      f"{len(GIDIAS_EXCERPTS):,} unrated species")

GIDIAS excerpts available as evidence for 1,175 unrated species


## Step 7 · Test the model on one species

One call, shown in full: the question, the raw answer and what
`parse_answer` made of it. If the parsed answer is `None`, the model does
not return the JSON format; try a more capable model or lower
`TEMPERATURE`. The test uses a well-studied invader, so a real model
should give high categories with plausible evidence.

In [16]:
print("Step 7: one test call")
test_species = {"taxon_name": "Rattus rattus", "kingdom": "Animalia",
                "class": "Mammalia", "family": "Muridae"}
test_prompt = build_prompt(test_species)
print(test_prompt, "\n--- answer ---")
test_answer = ask_model(SYSTEM_PROMPT, test_prompt)
print(test_answer, "\n--- parsed ---")
print(json.dumps(parse_answer(test_answer), indent=1))

Step 7: one test call
Species: Rattus rattus
Taxonomy (GBIF Backbone): kingdom: Animalia, class: Mammalia, family: Muridae

Report the documented impacts as specified. 
--- answer ---
{"knows_species": true, "eicat": {"category": "MV", "confidence": "low", "mechanisms": ["parasitism"], "affected_native_taxa": ["Plant"], "justification": "Made-up answer of the dry run."}, "seicat": {"category": "MO", "confidence": "low", "constituents": ["Freedom of choice and action"], "justification": "Made-up answer of the dry run."}, "ncp_negative": ["Materials, companionship and labor"], "realms": ["Terrestrial"], "countries": ["Nowhere"], "positive_nature": false, "positive_people": false, "evidence": []} 
--- parsed ---
{
 "knows_species": true,
 "eicat": {
  "category": "MV",
  "confidence": "low",
  "mechanisms": [
   "parasitism"
  ],
  "justification": "Made-up answer of the dry run.",
  "affected_native_taxa": [
   "Plant"
  ]
 },
 "seicat": {
  "category": "MO",
  "confidence": "low",
  "co

## Step 8 · Score the species

For every selected species the model is asked `N_REPEATS` times.

* Interrupting is safe: a re-run continues where it stopped.
* A species is asked again only if the model, the prompt or its evidence
  changed.
* The cache is the full record of what the model said.

A species for which no answer could be parsed stays without LLM data and
is listed after the loop.

In [17]:
print("Step 8: scoring")
new_llm, failed = (score_species(todo, "impact scoring") if len(todo)
                   else (pd.DataFrame(), []))
if failed:
    names_failed = todo.set_index("gbif_id").loc[failed, "taxon_name"]
    print(f"   no usable answer: {names_failed.tolist()[:15]}")
if len(new_llm):
    new_llm = new_llm.assign(impact_model=MODEL_LABEL,
                             impact_prompt_version=PROMPT_VERSION,
                             impact_date=date.today().isoformat())
kept_llm = old_llm[~old_llm["gbif_id"].isin(new_llm.get("gbif_id", []))]
llm = pd.concat([kept_llm, new_llm], ignore_index=True)
print(f"   LLM table: {len(llm):,} species "
      f"({len(new_llm):,} from this run)")
new_llm.head(8)[[c for c in ("taxon_name", "eicat", "eicat_runs", "seicat",
                             "seicat_runs", "impact_realms")
                 if c in new_llm.columns]]

Step 8: scoring


impact scoring:   0%|          | 0/97 [00:00<?, ?sp/s]

   impact scoring: 97 species scored, 0 failed, 291 new model call(s)
   LLM table: 97 species (97 from this run)


,taxon_name,eicat,eicat_runs,seicat,seicat_runs,impact_realms
0,Trechus rubens,DD,DD/DD/DD,DD,DD/DD/DD,
1,Attagenus unicolor,MN,MN/MN/MN,MC,MC/MC/MC,Marine
2,Ptinus tectus,MV,MV/MV/MV,MN,MN/MN/MN,Freshwater
3,Anarsia lineatella,DD,DD/DD/DD,DD,DD/DD/DD,
4,Elatobium abietinum,DD,DD/DD/DD,DD,DD/DD/DD,
5,Epinotia fraternana,MR,MR/MR/MR,DD,DD/DD/DD,Marine
6,Phytomyza gymnostoma,MO,MO/MO/MO,DD,DD/DD/DD,Marine
7,Saturnia pyri,MN,MN/MN/MN,MC,MC/MC/MC,Marine


## Step 9 · Benchmark the model against GIDIAS

`BENCHMARK_N` species with a GIDIAS EICAT category MC–MV are scored by
the model in exactly the same way, **without evidence**, and the two are
compared, for EICAT and (where GIDIAS has one) for SEICAT.

| Statistic | Meaning | What to look for |
|---|---|---|
| exact agreement | same category | — |
| within one category | same or neighbouring category | should be high |
| weighted kappa (quadratic) | agreement beyond chance, larger misses count more | below ~0.4 is weak |
| mean difference | LLM minus GIDIAS, in category steps | > 0: the model rates higher |
| answered DD | share of GIDIAS-rated species the model called data deficient | high: cautious, or knows little |

**Two caveats.** GIDIAS species are the studied ones and the model may
have read the same papers, so this is an **upper bound** for the obscure
species it is actually used on. And GIDIAS is not a gold standard: its
category is the maximum over single sources and it stores every local
extinction as MR, so a model answering MV is not necessarily wrong.

In [18]:
def weighted_kappa(a, b, n_levels):
    """Quadratic-weighted Cohen's kappa of two integer ratings."""
    a, b = np.asarray(a), np.asarray(b)
    observed = np.zeros((n_levels, n_levels))
    np.add.at(observed, (a, b), 1)
    observed /= observed.sum()
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0))
    i, j = np.indices((n_levels, n_levels))
    weights = (i - j) ** 2 / (n_levels - 1) ** 2
    denom = (weights * expected).sum()
    return float(1 - (weights * observed).sum() / denom) if denom else np.nan


def agreement_stats(reference, model):
    """Agreement of the model with a reference rating (both MC-MV / DD).

    Only rows with a reference category MC-MV are used.
    """
    use = reference.isin(EICAT_ORDER)
    reference, model = reference[use], model[use]
    rated = model.isin(EICAT_ORDER)
    if not len(reference) or not rated.any():
        return {"n_compared": int(len(reference)),
                "share_answered_dd": 1.0 if len(reference) else None}
    r = reference[rated].map(EICAT_ORDER.index).to_numpy()
    m = model[rated].map(EICAT_ORDER.index).to_numpy()
    return {"n_compared": int(len(reference)),
            "n_rated_by_llm": int(rated.sum()),
            "share_answered_dd": float((~rated).mean()),
            "exact_agreement": float((r == m).mean()),
            "within_one_category": float((np.abs(r - m) <= 1).mean()),
            "weighted_kappa": weighted_kappa(r, m, len(EICAT_ORDER)),
            "mean_difference_llm_minus_gidias": float((m - r).mean())}


print("Step 9: benchmark")
benchmark, agreement = pd.DataFrame(), {}
pool = gidias[gidias["eicat"].isin(EICAT_ORDER)]
if BENCHMARK_N <= 0 or len(pool) < 10:
    print("   skipped (BENCHMARK_N = 0 or fewer than 10 rated species)")
else:
    sample = pool.sample(min(BENCHMARK_N, len(pool)),
                         random_state=BENCHMARK_SEED)
    taxonomy = scans.drop_duplicates("gbif_id").set_index("gbif_id")[
        [c for c in ("class", "family") if c in scans.columns]]
    sample = sample[["gbif_id", "taxon_name", "kingdom", "eicat",
                     "seicat"]].join(taxonomy, on="gbif_id")
    scored, _ = score_species(sample, "benchmark", use_evidence=False)
    if len(scored):
        benchmark = scored.merge(
            sample[["gbif_id", "eicat", "seicat"]], on="gbif_id",
            suffixes=("_llm", "_gidias"))
        agreement = {
            "eicat": agreement_stats(benchmark["eicat_gidias"],
                                     benchmark["eicat_llm"]),
            "seicat": agreement_stats(benchmark["seicat_gidias"],
                                      benchmark["seicat_llm"])}
        print(pd.DataFrame(agreement).round(3).to_string())
        print("\n   EICAT; rows: GIDIAS, columns: LLM")
        print(pd.crosstab(
            pd.Categorical(benchmark["eicat_gidias"], EICAT_ORDER),
            pd.Categorical(benchmark["eicat_llm"], VALID),
            rownames=["GIDIAS"], colnames=["LLM"], dropna=False).to_string())
if DRY_RUN and agreement:
    print("\n   (dry run: these numbers come from made-up answers)")

Step 9: benchmark


benchmark:   0%|          | 0/100 [00:00<?, ?sp/s]

   benchmark: 100 species scored, 0 failed, 300 new model call(s)
                                    eicat  seicat
n_compared                        100.000  18.000
n_rated_by_llm                     81.000   8.000
share_answered_dd                   0.190   0.556
exact_agreement                     0.247   0.250
within_one_category                 0.630   0.625
weighted_kappa                      0.090   0.091
mean_difference_llm_minus_gidias    0.407  -0.375

   EICAT; rows: GIDIAS, columns: LLM
LLM     MC  MN  MO  MR  MV  DD
GIDIAS                        
MC       3   1   0   1   0   2
MN       5   5   6   9   3   5
MO       4   5  12   9   9   9
MR       4   0   3   0   2   2
MV       0   0   0   0   0   1

   (dry run: these numbers come from made-up answers)


## Step 10 · Merge both sources into the master file

Per species, block by block:

| Block | Columns | Taken from |
|---|---|---|
| EICAT | `eicat`, confidence, agreement, mechanism, affected taxa, evidence counts, countries, justification, references | GIDIAS if it has MC–MV; else the LLM if it has MC–MV; else whichever has DD (GIDIAS first) → `eicat_source` |
| SEICAT | `seicat`, confidence, agreement, constituents, evidence counts, justification | the same rule → `seicat_source` |
| Context | realms, countries, positive impacts, NCP categories | GIDIAS if it has impact records for the species; else the LLM → `context_source` |

`impact_source` lists the sources used in the row (`gidias`, `llm` or
`gidias+llm`). `impact_model`, `impact_prompt_version`, `impact_date`
and `impact_n_runs` are filled when the LLM contributed. A species
without any row is "not evaluated" (NE) in the viewer.

In a dry run the made-up LLM answers are left out here.

In [19]:
EICAT_BLOCK = ["eicat", "eicat_confidence", "eicat_agreement",
               "eicat_mechanism", "affected_native_taxa", "eicat_n_records",
               "eicat_n_sources", "eicat_countries", "eicat_justification",
               "eicat_reference"]
SEICAT_BLOCK = ["seicat", "seicat_confidence", "seicat_agreement",
                "seicat_constituents", "seicat_n_records",
                "seicat_n_sources", "seicat_justification"]
CONTEXT_BLOCK = ["impact_realms", "impact_countries", "impact_n_countries",
                 "impact_iso3", "positive_nature", "positive_people",
                 "ncp_categories"]
PROVENANCE = ["impact_model", "impact_prompt_version", "impact_date",
              "impact_n_runs", "llm_has_evidence"]
MASTER_FIRST = (["gbif_id", "taxon_name", "kingdom", "impact_source"]
                + EICAT_BLOCK[:1] + ["eicat_source"] + EICAT_BLOCK[1:]
                + ["global_extinction"]
                + SEICAT_BLOCK[:1] + ["seicat_source"] + SEICAT_BLOCK[1:]
                + ["context_source"] + CONTEXT_BLOCK
                + ["impact_n_records", "impact_n_sources"] + PROVENANCE
                + ["eicat_expert"])


def pick_block(g_frame, l_frame, columns, use_g, use_l):
    """Columns of one block: GIDIAS where ``use_g``, LLM where ``use_l``."""
    out = {}
    for col in columns:
        g_col = g_frame.get(col, pd.Series(np.nan, index=g_frame.index))
        l_col = l_frame.get(col, pd.Series(np.nan, index=l_frame.index))
        out[col] = g_col.where(use_g, l_col.where(use_l)).astype(object)
    return pd.DataFrame(out)


def merge_impacts(gidias_table, llm_table):
    """Master table from the GIDIAS and the LLM table (see the rules).

    Returns
    -------
    pandas.DataFrame
        One row per species of either table, MASTER_FIRST columns first.
    """
    g = gidias_table.set_index("gbif_id")
    lt = llm_table.set_index("gbif_id")
    for col in ("taxon_name", "kingdom", "eicat", "seicat"):
        if col not in lt.columns:  # no LLM answers at all
            lt[col] = np.nan
    index = g.index.union(lt.index)
    g, lt = g.reindex(index), lt.reindex(index)
    has_g, has_l = g["eicat"].notna(), lt["eicat"].notna()

    def choose(column):
        g_rated = g[column].isin(EICAT_ORDER)
        l_rated = lt[column].isin(EICAT_ORDER)
        use_g = g_rated | (has_g & ~l_rated & g[column].eq("DD"))
        use_l = ~use_g & (l_rated | lt[column].eq("DD"))
        return use_g, use_l

    out = pd.DataFrame(index=index)
    out["taxon_name"] = g["taxon_name"].where(has_g, lt["taxon_name"])
    out["kingdom"] = g["kingdom"].where(has_g, lt["kingdom"])
    sources = {}
    for name, column, block in (("eicat", "eicat", EICAT_BLOCK),
                                ("seicat", "seicat", SEICAT_BLOCK)):
        use_g, use_l = choose(column)
        out = out.join(pick_block(g, lt, block, use_g, use_l))
        sources[name] = pd.Series(
            np.select([use_g, use_l], ["gidias", "llm"], default="none"),
            index=index)
        out[f"{name}_source"] = sources[name]
    out["eicat"] = out["eicat"].fillna("DD")
    out.loc[out["eicat_source"] == "none", "eicat_source"] = np.where(
        has_g[out["eicat_source"] == "none"], "gidias", "llm")
    ctx_g = has_g & (g["impact_n_records"].fillna(0) > 0)
    ctx_l = ~ctx_g & has_l
    out = out.join(pick_block(g, lt, CONTEXT_BLOCK, ctx_g, ctx_l))
    out["context_source"] = np.select([ctx_g, ctx_l], ["gidias", "llm"],
                                      default="none")
    used = pd.concat([out["eicat_source"], out["seicat_source"],
                      out["context_source"]], axis=1)
    out["impact_source"] = ["+".join(s for s in ("gidias", "llm")
                                     if s in set(row))
                            for row in used.to_numpy()]
    llm_used = used.eq("llm").any(axis=1)
    for col in PROVENANCE:
        out[col] = lt.get(col, pd.Series(np.nan, index=index)).where(
            llm_used)
    rest = [c for c in g.columns if c not in out.columns]
    out = out.join(g[rest]).reset_index().rename(
        columns={"index": "gbif_id"})
    out["eicat_expert"] = ""
    return out[[c for c in MASTER_FIRST if c in out.columns]
               + [c for c in out.columns if c not in MASTER_FIRST]]


print("Step 10: master table")
llm_for_master = llm.iloc[0:0] if DRY_RUN else llm
master = merge_impacts(gidias, llm_for_master)

# --- published IUCN assessments as a reference column (optional)
if EXPERT_CSV.exists():
    expert = pd.read_csv(EXPERT_CSV, dtype=str, keep_default_na=False)
    if "eicat_source" in expert.columns:
        expert = expert[expert["eicat_source"].str.lower()
                        .isin(["expert", ""])]
    expert = expert[expert["eicat"].str.upper().isin(VALID)]
    ref = dict(zip(pd.to_numeric(expert["gbif_id"], errors="coerce"),
                   expert["eicat"].str.upper()))
    master["eicat_expert"] = master["gbif_id"].map(ref).fillna("")
    print(f"   {int((master['eicat_expert'] != '').sum()):,} species "
          "with a published assessment in eicat_expert")

for col in ("eicat_n_records", "eicat_n_sources", "seicat_n_records",
            "seicat_n_sources", "impact_n_countries", "impact_n_records",
            "impact_n_sources", "impact_n_runs"):
    master[col] = pd.to_numeric(master[col], errors="coerce").astype("Int64")
master["seicat"] = master["seicat"].fillna("")
print(f"   {len(master):,} species | by source: "
      f"{master['impact_source'].value_counts().to_dict()}")
print("   EICAT by source:")
print(pd.crosstab(master["eicat_source"], master["eicat"])
      .reindex(columns=VALID, fill_value=0).to_string())
print("   SEICAT by source:")
print(pd.crosstab(master["seicat_source"],
                  master["seicat"].replace("", "none"))
      .reindex(columns=VALID + ["none"], fill_value=0).to_string())

Step 10: master table
   3,187 species | by source: {'gidias': 3187}
   EICAT by source:
eicat         MC   MN   MO   MR  MV    DD
eicat_source                             
gidias        64  396  558  198  10  1961
   SEICAT by source:
seicat         MC   MN   MO  MR  MV  DD  none
seicat_source                                
gidias         36  176  126  40   0   0     0
none            0    0    0   0   0   0  2809


## Step 11 · Quality checks

* **Name matching:** share of GIDIAS names and records with a species
  key.
* **Coverage of the lists:** how many top-`TOP_X` species now have an
  EICAT and a SEICAT category, and from which source.
* **Single-source MR and MV** from GIDIAS: one publication sets the
  category; read these first.
* **LLM answers:** distribution (a model that gives MR or MV to many
  little-known species is rating reputation, not evidence), agreement
  between runs, and how often it marked a source as unverified. Read the
  justification of every LLM-derived MR and MV.

In [20]:
print("Step 11: QA")
in_top = master[master["gbif_id"].isin(best["gbif_id"])]
coverage = {
    "top_species": int(len(best)),
    "eicat_rated": in_top.loc[in_top["eicat"].isin(EICAT_ORDER),
                              "eicat_source"].value_counts().to_dict(),
    "seicat_rated": in_top.loc[in_top["seicat"].isin(EICAT_ORDER),
                               "seicat_source"].value_counts().to_dict(),
}
single = gidias[gidias["eicat"].isin(["MR", "MV"])
                & (gidias["eicat_n_sources"] == 1)]
summary = {
    "date": date.today().isoformat(),
    "gidias_version": GIDIAS_VERSION, "gidias_file": GIDIAS_CSV.name,
    "settings": {"NATIVE_TAXA_ONLY": NATIVE_TAXA_ONLY,
                 "LEVEL3_CATEGORY": LEVEL3_CATEGORY,
                 "MC_DIRECTIONS": list(MC_DIRECTIONS),
                 "MIN_FUZZY_CONFIDENCE": MIN_FUZZY_CONFIDENCE,
                 "FILL_GAPS": list(FILL_GAPS)},
    "gidias": {
        "rows": int(len(raw)), "names": int(len(names)),
        "names_matched_share": float(matched.mean()),
        "records_matched_share": float(
            names.loc[matched, "n_records"].sum() / len(g)),
        "species": int(len(gidias)),
        "eicat": gidias["eicat"].value_counts().to_dict(),
        "seicat": gidias.loc[gidias["seicat"] != "", "seicat"]
        .value_counts().to_dict(),
        "single_source_mr_mv": int(len(single))},
    "llm": None if DRY_RUN else {
        "model": MODEL_LABEL, "provider": PROVIDER,
        "prompt_version": PROMPT_VERSION, "n_repeats": N_REPEATS,
        "temperature": TEMPERATURE, "evidence": EVIDENCE, "top_x": TOP_X,
        "species_scored_this_run": int(len(new_llm)),
        "species_failed_this_run": int(len(failed)),
        "species_in_table": int(len(llm)),
        "agreement_with_gidias": agreement or None},
    "master": {
        "species": int(len(master)),
        "by_source": master["impact_source"].value_counts().to_dict(),
        "eicat_by_source": master.loc[master["eicat"].isin(EICAT_ORDER),
                                      "eicat_source"]
        .value_counts().to_dict(),
        "seicat_by_source": master.loc[master["seicat"].isin(EICAT_ORDER),
                                       "seicat_source"]
        .value_counts().to_dict()},
    "coverage_of_lists": coverage,
    "note": ("Evidence summaries (GIDIAS) and model proposals (LLM); "
             "neither is an IUCN EICAT or SEICAT assessment. Not used by "
             "the risk model."),
}
print(f"   names matched: {matched.mean():.1%} | records matched: "
      f"{summary['gidias']['records_matched_share']:.1%}")
print(f"   top-{TOP_X} species: {coverage['top_species']:,} | EICAT MC-MV "
      f"{coverage['eicat_rated']} | SEICAT MC-MV {coverage['seicat_rated']}")
if len(new_llm):
    unverified = new_llm["eicat_reference"].str.contains("unverified",
                                                         case=False)
    print("   LLM EICAT:", new_llm["eicat"].value_counts().to_dict(),
          "| SEICAT:", new_llm["seicat"].value_counts().to_dict())
    print(f"   all runs agree (EICAT): "
          f"{(new_llm['eicat_agreement'] == 1).mean():.0%} | with supplied "
          f"evidence: {new_llm['llm_has_evidence'].mean():.0%} | with an "
          f"'unverified' source: {unverified.mean():.0%}")
    llm_eicat = set(master.loc[master["eicat_source"] == "llm", "gbif_id"])
    high = new_llm[new_llm["eicat"].isin(["MR", "MV"])
                   & new_llm["gbif_id"].isin(llm_eicat)]
    print(f"\n   LLM Major / Massive ({len(high)}): read these")
    print(high[["taxon_name", "eicat", "eicat_runs",
                "eicat_justification"]].head(25).to_string(index=False))
print(f"\n   GIDIAS MR / MV resting on one source ({len(single)}):")
single.head(15)[["taxon_name", "eicat", "eicat_mechanism",
                 "eicat_countries", "eicat_reference"]]

Step 11: QA
   names matched: 94.6% | records matched: 96.2%
   top-100 species: 100 | EICAT MC-MV {'gidias': 6} | SEICAT MC-MV {'gidias': 3}
   LLM EICAT: {'DD': 21, 'MN': 20, 'MO': 17, 'MC': 14, 'MR': 13, 'MV': 12} | SEICAT: {'DD': 42, 'MR': 14, 'MV': 12, 'MN': 11, 'MC': 9, 'MO': 9}
   all runs agree (EICAT): 100% | with supplied evidence: 19% | with an 'unverified' source: 0%

   LLM Major / Massive (0): read these
Empty DataFrame
Columns: [taxon_name, eicat, eicat_runs, eicat_justification]
Index: []

   GIDIAS MR / MV resting on one source (113):


,taxon_name,eicat,eicat_mechanism,eicat_countries,eicat_reference
127,Polygraphus proximus,MR,grazing/herbivory/browsing,Russia,Krivets S.A. Bisirova E.M. Kerchev I.A. Pats E.N. Chernova N.A. TRANSFORMATION OF TAIG...
193,Monomorium monomorium,MR,predation,USA,LaPolla JS Otte D Spearman LA. 2000. Assessment of the effects of ants on Hawaiian cri...
197,Monomorium floricola,MR,,Ecuador,Traveset A Heleno R Chamorro S Vargas P McMullen CK Castro-Urgal R Nogales M Herrera H...
205,Tapinoma melanocephalum,MR,,Ecuador,Traveset A Heleno R Chamorro S Vargas P McMullen CK Castro-Urgal R Nogales M Herrera H...
222,Pachycondyla chinensis,MR,competition,USA,Guénard B Dunn RR. 2010. A new (Old) invasive ant in the hardwood forests of eastern N...
224,Technomyrmex albipes,MR,indirect impacts through interactions with other species,Fiji,Thaman RR. 2018. 1 The 2016 Fiji Ant-Mealybug Bioinvasion: : threat to food health liv...
246,Tetramorium simillimum,MR,competition,Malaysia,Pfeiffer M Brühl C. 2007. Low species richness and high impact of invasive species in ...
258,Paratrechina bourbonica,MR,,USA,Bleil R Blüthgen N Junker RR. 2011. Ant-plant mutualism in Hawai‘i? Invasive ants redu...
259,Paratrechina fulva,MR,competition; predation,USA,LeBrun EG Abbott J Gilbert LE. 2013. Imported crazy ant displaces imported fire ant re...
268,Bombus terrestris,MR,competition,Japan,10.1016/j.biocon.2008.07.029


## Step 12 · Write outputs

1. **`species_impacts_master.csv`**, and a copy in
   `model_outputs/exports/` (`COPY_TO_EXPORTS`) so the viewer finds it
   when that folder is imported. The lists themselves are **not**
   changed.
2. `gidias_impact_records.csv` (list columns joined with `; `) and the
   name-matching audit.
3. The LLM table, the benchmark and the summary. The summary is also
   added to `model_card.json` as `impacts`, for the viewer's "About"
   tab.

In a dry run the LLM table and the benchmark are not written, so made-up
answers never reach a file that is used.

In [21]:
print("Step 12: writing")
records_out = records.drop(columns=["source_id", "text_excerpt"]).copy()
for col in ("countries", "iso3", "units_of_analysis", "mechanisms",
            "ncp_categories", "cwb_constituents"):
    records_out[col] = records_out[col].map("; ".join)
first = ["gbif_id", "gidias_name", "eicat_record", "seicat_record",
         "positive_level"]
records_out = records_out[first + [c for c in records_out.columns
                                   if c not in first]]
outputs = [(master, MASTER_OUT), (records_out, RECORDS_OUT),
           (names.sort_values("n_records", ascending=False),
            MATCH_AUDIT_OUT)]
if not DRY_RUN:
    outputs.append((llm, LLM_SCORES))
    if len(benchmark):
        BENCHMARK_OUT.parent.mkdir(parents=True, exist_ok=True)
        outputs.append((benchmark, BENCHMARK_OUT))
for frame, path in outputs:
    backup(path)
    frame.to_csv(path, index=False)
    print(f"   {path.name}: {len(frame):,} rows")
SUMMARY_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(SUMMARY_OUT, "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"   {SUMMARY_OUT.name}")

if COPY_TO_EXPORTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(MASTER_OUT, EXPORT_DIR / MASTER_NAME)
    print(f"   copied {MASTER_NAME} to {EXPORT_DIR}")
    card_path = EXPORT_DIR / "model_card.json"
    if card_path.exists():
        with open(card_path) as fh:
            card = json.load(fh)
        card["impacts"] = summary
        card.pop("eicat_llm", None)  # entry of the former notebook 07
        with open(card_path, "w") as fh:
            json.dump(card, fh, indent=2, default=str)
        print(f"   {card_path.name}: impacts entry added")
print("DONE")

Step 12: writing
   species_impacts_master.csv: 3,187 rows
   gidias_impact_records.csv: 22,858 rows
   gidias_name_matching_audit.csv: 3,384 rows
   impacts_summary.json
   copied species_impacts_master.csv to c:\Users\simon\Documents\GitHub\horizon-scanner\data\model_outputs\exports
   model_card.json: impacts entry added
DONE
